# 1. Pre-baseline fixed equal-fusion cross-validation aggregation

**Experiment name:** `temporal_prebaseline_fixed_equal_fusion_md050`

This notebook consolidates the held-out test results from all five outer folds of the pre-baseline temporal-sensitivity experiment with fixed 50/50 fusion for MCI prognosis.

The availability-gated cross-modal interaction architecture is retained. The participant-specific learned reliability gate is replaced by fixed pathway weights of 0.50 for interaction pathway and 0.50 for evidence pathway. Each participant contributes exactly one out-of-fold prediction.

The principal result is the pooled out-of-fold ROC AUC across all 544 MCI participants. The notebook also reports fold-level mean ± standard deviation, average precision, threshold-dependent performance, probability calibration, uncertainty behaviour, risk-coverage performance, bootstrap confidence intervals, and modality-availability analyses.


In [ ]:
# ============================================================
# 1. Connecting to Google Drive and defining the pre-baseline temporal-sensitivity experiment paths
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Connect to Google Drive when running in Colab
# ------------------------------------------------------------

try:
    from google.colab import drive

    # I mount Google Drive only when it is not already mounted.
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")

except ImportError:
    print(
        "Google Colab is not detected. "
        "I will use the paths as currently available."
    )


# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/adni_mri"
)

MODEL_ROOT = (
    PROJECT_ROOT
    / "models"
    / "3mt_tmc_evidential"
)

EXPERIMENT_NAME = "temporal_prebaseline_fixed_equal_fusion_md050"
TASK_NAME = "mci_prognosis"

EXPERIMENT_ROOT = (
    MODEL_ROOT
    / "experiments"
    / EXPERIMENT_NAME
)

TASK_EXPERIMENT_ROOT = (
    EXPERIMENT_ROOT
    / TASK_NAME
)


# ------------------------------------------------------------
# Cross-validation folds
# ------------------------------------------------------------

OUTER_FOLDS = [
    0,
    1,
    2,
    3,
    4,
]


# ------------------------------------------------------------
# Final aggregation-output folder
# ------------------------------------------------------------

AGGREGATION_ROOT = (
    EXPERIMENT_ROOT
    / "cross_validation_aggregation"
    / TASK_NAME
)

TABLE_DIR = (
    AGGREGATION_ROOT
    / "tables"
)

FIGURE_DIR = (
    AGGREGATION_ROOT
    / "figures"
)

POOLED_PREDICTION_DIR = (
    AGGREGATION_ROOT
    / "pooled_predictions"
)

STATISTICS_DIR = (
    AGGREGATION_ROOT
    / "statistics"
)


# I create new aggregation folders without modifying
# any of the existing fold-specific training outputs.
for directory in [
    AGGREGATION_ROOT,
    TABLE_DIR,
    FIGURE_DIR,
    POOLED_PREDICTION_DIR,
    STATISTICS_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ------------------------------------------------------------
# Expected files for every fold
# ------------------------------------------------------------

fold_paths = {}

for fold in OUTER_FOLDS:

    fold_root = (
        TASK_EXPERIMENT_ROOT
        / f"fold_{fold}"
    )

    fold_paths[fold] = {
        "fold_root":
            fold_root,

        "best_checkpoint":
            (
                fold_root
                / "checkpoints"
                / "best_validation_auc_checkpoint.pt"
            ),

        "last_checkpoint":
            (
                fold_root
                / "checkpoints"
                / "last_epoch_checkpoint.pt"
            ),

        "training_history":
            (
                fold_root
                / "history"
                / "training_history.csv"
            ),

        "training_configuration":
            (
                fold_root
                / "history"
                / "training_configuration.json"
            ),

        "test_predictions":
            (
                fold_root
                / "predictions"
                / "test_predictions.csv"
            ),

        "test_metrics":
            (
                fold_root
                / "predictions"
                / "test_metrics.json"
            ),

        "test_metrics_summary":
            (
                fold_root
                / "predictions"
                / "test_metrics_summary.csv"
            ),
    }


# ------------------------------------------------------------
# Check the required aggregation inputs
# ------------------------------------------------------------

required_file_names = [
    "best_checkpoint",
    "training_history",
    "test_predictions",
    "test_metrics",
    "test_metrics_summary",
]

inventory_rows = []

for fold in OUTER_FOLDS:

    for file_name in required_file_names:

        file_path = fold_paths[
            fold
        ][
            file_name
        ]

        inventory_rows.append(
            {
                "FOLD":
                    fold,

                "FILE_TYPE":
                    file_name,

                "EXISTS":
                    file_path.exists(),

                "PATH":
                    str(
                        file_path
                    ),
            }
        )


fold_output_inventory = pd.DataFrame(
    inventory_rows
)


# ------------------------------------------------------------
# Stop if any required fold output is missing
# ------------------------------------------------------------

missing_outputs = fold_output_inventory.loc[
    ~fold_output_inventory[
        "EXISTS"
    ]
].copy()


print("=" * 72)
print("FIXED-EQUAL-FUSION CROSS-VALIDATION AGGREGATION SETUP")
print("=" * 72)

print(
    f"\nExperiment: {EXPERIMENT_NAME}"
)

print(
    f"Task: {TASK_NAME}"
)

print(
    "Outer folds: "
    + ", ".join(
        str(fold)
        for fold in OUTER_FOLDS
    )
)

print(
    f"\nFold-output root:\n{TASK_EXPERIMENT_ROOT}"
)

print(
    f"\nNew aggregation-output root:\n{AGGREGATION_ROOT}"
)


print(
    "\nRequired fold-output inventory:"
)

display(
    fold_output_inventory
)


if not missing_outputs.empty:

    print(
        "\nMissing required outputs:"
    )

    display(
        missing_outputs
    )

    raise FileNotFoundError(
        "At least one required cross-validation output "
        "is missing. Aggregation cannot begin."
    )


print(
    "\nAll required outputs were found for folds 0--4."
)

print(
    "\nThe original fold directories are read-only inputs "
    "in this notebook."
)

print(
    "All new tables, figures, pooled predictions, and "
    "statistical outputs will be written only under:"
)

print(
    AGGREGATION_ROOT
)

## 1.1. Loading the five fold results

This section loads the saved test metrics and participant-level test predictions from folds 0-4 of the pre-baseline temporal-sensitivity experiment. The fold-specific training outputs are treated as read-only inputs.


In [ ]:
# ============================================================
# 2. Loading fold-level metrics and out-of-fold predictions
# ============================================================

# ------------------------------------------------------------
# Containers for the five folds
# ------------------------------------------------------------

fold_metric_summaries = []
fold_complete_metrics = []
fold_prediction_tables = []


# ------------------------------------------------------------
# Load every fold
# ------------------------------------------------------------

for fold in OUTER_FOLDS:

    print(
        f"Loading fold {fold}..."
    )


    # --------------------------------------------------------
    # Load the compact model-output comparison table
    # --------------------------------------------------------

    fold_metric_summary = pd.read_csv(
        fold_paths[
            fold
        ][
            "test_metrics_summary"
        ]
    )

    fold_metric_summary.insert(
        0,
        "FOLD",
        fold,
    )

    fold_metric_summaries.append(
        fold_metric_summary
    )


    # --------------------------------------------------------
    # Load the complete JSON evaluation record
    # --------------------------------------------------------

    with open(
        fold_paths[
            fold
        ][
            "test_metrics"
        ],
        "r",
        encoding="utf-8",
    ) as metrics_file:

        fold_metrics_json = json.load(
            metrics_file
        )


    fold_complete_metrics.append(
        {
            "FOLD":
                fold,

            "TASK":
                fold_metrics_json[
                    "task"
                ],

            "CHECKPOINT_EPOCH":
                fold_metrics_json[
                    "checkpoint_epoch"
                ],

            "CHECKPOINT_VALIDATION_AUC":
                fold_metrics_json[
                    "checkpoint_validation_auc"
                ],

            "CLASSIFICATION_THRESHOLD":
                fold_metrics_json[
                    "classification_threshold"
                ],

            "TEST_PARTICIPANTS":
                fold_metrics_json[
                    "test_participants"
                ],

            "TEST_BATCHES":
                fold_metrics_json[
                    "test_batches"
                ],

            "MAXIMUM_MODALITY_COUNT_DIFFERENCE":
                fold_metrics_json[
                    "maximum_modality_count_difference"
                ],

            "EVALUATED_AT":
                fold_metrics_json[
                    "evaluated_at"
                ],
        }
    )


    # --------------------------------------------------------
    # Load participant-level test predictions
    # --------------------------------------------------------

    fold_predictions = pd.read_csv(
        fold_paths[
            fold
        ][
            "test_predictions"
        ]
    )

    fold_predictions.insert(
        0,
        "FOLD",
        fold,
    )

    fold_prediction_tables.append(
        fold_predictions
    )


# ------------------------------------------------------------
# Combine fold-level files
# ------------------------------------------------------------

all_fold_metric_summaries = pd.concat(
    fold_metric_summaries,
    ignore_index=True,
)

all_fold_complete_metrics = pd.DataFrame(
    fold_complete_metrics
)

pooled_out_of_fold_predictions = pd.concat(
    fold_prediction_tables,
    ignore_index=True,
)


# ------------------------------------------------------------
# Standardise model-output labels
# ------------------------------------------------------------

all_fold_metric_summaries[
    "OUTPUT"
] = (
    all_fold_metric_summaries[
        "OUTPUT"
    ]
    .astype(
        "string"
    )
    .str.strip()
)


# ------------------------------------------------------------
# Basic validation expectations
# ------------------------------------------------------------

EXPECTED_OUTPUTS = {
    "Hybrid",
    "3MT-only",
    "TMC-only",
}

EXPECTED_FOLD_COUNT = len(
    OUTER_FOLDS
)

EXPECTED_METRIC_ROWS = (
    EXPECTED_FOLD_COUNT
    * len(
        EXPECTED_OUTPUTS
    )
)

EXPECTED_TOTAL_PARTICIPANTS = 544


# ------------------------------------------------------------
# Check model-output coverage by fold
# ------------------------------------------------------------

output_coverage = (
    all_fold_metric_summaries
    .groupby(
        "FOLD"
    )[
        "OUTPUT"
    ]
    .agg(
        lambda values:
            sorted(
                values.tolist()
            )
    )
    .reset_index(
        name="OUTPUTS"
    )
)


invalid_output_folds = []

for _, row in output_coverage.iterrows():

    if set(
        row[
            "OUTPUTS"
        ]
    ) != EXPECTED_OUTPUTS:

        invalid_output_folds.append(
            int(
                row[
                    "FOLD"
                ]
            )
        )


# ------------------------------------------------------------
# Check participant counts
# ------------------------------------------------------------

fold_prediction_counts = (
    pooled_out_of_fold_predictions
    .groupby(
        "FOLD"
    )
    .size()
    .reset_index(
        name="PREDICTION_ROWS"
    )
)


fold_prediction_counts = (
    fold_prediction_counts
    .merge(
        all_fold_complete_metrics[
            [
                "FOLD",
                "TEST_PARTICIPANTS",
            ]
        ],
        on="FOLD",
        how="left",
        validate="one_to_one",
    )
)


fold_prediction_counts[
    "COUNT_MATCH"
] = (
    fold_prediction_counts[
        "PREDICTION_ROWS"
    ]
    == fold_prediction_counts[
        "TEST_PARTICIPANTS"
    ]
)


# ------------------------------------------------------------
# Identify the participant identifier column
# ------------------------------------------------------------

possible_participant_identifier_columns = [
    "RID",
    "rid",
    "PARTICIPANT_ID",
    "participant_id",
]


participant_identifier_column = next(
    (
        column
        for column in (
            possible_participant_identifier_columns
        )
        if column in (
            pooled_out_of_fold_predictions.columns
        )
    ),
    None,
)


if participant_identifier_column is None:

    raise KeyError(
        "No recognised participant identifier column was found "
        "in the pooled test-prediction files."
    )


# ------------------------------------------------------------
# Check that every participant appears in one test fold only
# ------------------------------------------------------------

participant_fold_counts = (
    pooled_out_of_fold_predictions
    .groupby(
        participant_identifier_column
    )[
        "FOLD"
    ]
    .nunique()
)


participants_in_multiple_folds = (
    participant_fold_counts[
        participant_fold_counts
        > 1
    ]
)


duplicated_prediction_rows = (
    pooled_out_of_fold_predictions
    .duplicated(
        subset=[
            participant_identifier_column
        ],
        keep=False,
    )
)


duplicated_participants = (
    pooled_out_of_fold_predictions
    .loc[
        duplicated_prediction_rows,
        [
            "FOLD",
            participant_identifier_column,
        ],
    ]
    .sort_values(
        [
            participant_identifier_column,
            "FOLD",
        ]
    )
)


# ------------------------------------------------------------
# Check evaluation integrity
# ------------------------------------------------------------

wrong_task_rows = (
    all_fold_complete_metrics[
        all_fold_complete_metrics[
            "TASK"
        ]
        != TASK_NAME
    ]
)


nonzero_modality_differences = (
    all_fold_complete_metrics[
        all_fold_complete_metrics[
            "MAXIMUM_MODALITY_COUNT_DIFFERENCE"
        ]
        != 0
    ]
)


threshold_values = sorted(
    all_fold_complete_metrics[
        "CLASSIFICATION_THRESHOLD"
    ]
    .unique()
    .tolist()
)


# ------------------------------------------------------------
# Save the loaded combined inputs
# ------------------------------------------------------------

ALL_FOLD_METRICS_PATH = (
    TABLE_DIR
    / "all_fold_test_metrics.csv"
)

ALL_FOLD_METADATA_PATH = (
    TABLE_DIR
    / "all_fold_test_metadata.csv"
)

POOLED_RAW_PREDICTIONS_PATH = (
    POOLED_PREDICTION_DIR
    / "pooled_out_of_fold_predictions_raw.csv"
)


all_fold_metric_summaries.to_csv(
    ALL_FOLD_METRICS_PATH,
    index=False,
)

all_fold_complete_metrics.to_csv(
    ALL_FOLD_METADATA_PATH,
    index=False,
)

pooled_out_of_fold_predictions.to_csv(
    POOLED_RAW_PREDICTIONS_PATH,
    index=False,
)


# ------------------------------------------------------------
# Display validation results
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("FOLD OUTPUTS LOADED")
print("=" * 72)


print(
    "\nFold-level test metadata:"
)

display(
    all_fold_complete_metrics
)


print(
    "\nModel-output coverage:"
)

display(
    output_coverage
)


print(
    "\nPrediction counts by fold:"
)

display(
    fold_prediction_counts
)


print(
    "\nCombined metric rows: "
    f"{len(all_fold_metric_summaries)}"
)

print(
    "Expected metric rows: "
    f"{EXPECTED_METRIC_ROWS}"
)

print(
    "\nTotal pooled prediction rows: "
    f"{len(pooled_out_of_fold_predictions)}"
)

print(
    "Expected MCI-prognosis participants: "
    f"{EXPECTED_TOTAL_PARTICIPANTS}"
)

print(
    "Unique participant identifiers: "
    f"{pooled_out_of_fold_predictions[participant_identifier_column].nunique()}"
)

print(
    "\nParticipant identifier column: "
    f"{participant_identifier_column}"
)

print(
    "Classification thresholds found: "
    f"{threshold_values}"
)


# ------------------------------------------------------------
# Stop if an integrity check fails
# ------------------------------------------------------------

if len(
    all_fold_metric_summaries
) != EXPECTED_METRIC_ROWS:

    raise ValueError(
        "The combined fold-metric table does not contain "
        "the expected 15 rows."
    )


if invalid_output_folds:

    raise ValueError(
        "One or more folds do not contain all three outputs: "
        f"{invalid_output_folds}"
    )


if not fold_prediction_counts[
    "COUNT_MATCH"
].all():

    raise ValueError(
        "At least one fold's prediction-row count does not "
        "match the participant count stored in test_metrics.json."
    )


if len(
    pooled_out_of_fold_predictions
) != EXPECTED_TOTAL_PARTICIPANTS:

    raise ValueError(
        "The pooled out-of-fold prediction count does not match "
        f"the expected cohort size of {EXPECTED_TOTAL_PARTICIPANTS}."
    )


if not participants_in_multiple_folds.empty:

    raise ValueError(
        "At least one participant appears in more than one "
        "outer test fold."
    )


if not duplicated_participants.empty:

    print(
        "\nDuplicated participant prediction rows:"
    )

    display(
        duplicated_participants
    )

    raise ValueError(
        "At least one participant has more than one out-of-fold "
        "prediction row."
    )


if not wrong_task_rows.empty:

    raise ValueError(
        "At least one fold contains metrics for a task other than "
        f"{TASK_NAME}."
    )


if not nonzero_modality_differences.empty:

    raise ValueError(
        "At least one fold used effective test modality counts "
        "different from the original modality counts."
    )


if threshold_values != [
    0.5
]:

    raise ValueError(
        "The folds do not all use the fixed classification "
        "threshold of 0.50."
    )


print(
    "\nAll five fold outputs were loaded successfully."
)

print(
    "Every participant has exactly one out-of-fold test prediction."
)

print(
    "All folds used the fixed classification threshold of 0.50."
)

print(
    "No test-time modality dropout was detected."
)


print(
    "\nCombined fold metrics saved to:\n"
    f"{ALL_FOLD_METRICS_PATH}"
)

print(
    "\nCombined fold metadata saved to:\n"
    f"{ALL_FOLD_METADATA_PATH}"
)

print(
    "\nRaw pooled out-of-fold predictions saved to:\n"
    f"{POOLED_RAW_PREDICTIONS_PATH}"
)

### 1.1.1. Fold-output integrity check

The checks below confirm that all five folds are present, that their held-out test sets contain the expected 544 participants in total, and that every participant contributes exactly one out-of-fold prediction.


## 1.2. Calculating fold-level cross-validation summary statistics

summarise the test performance across the five outer folds.

For each model output, The notebook calculates the fold-level:

- mean;
- sample standard deviation;
- minimum;
- maximum.

The three evaluated outputs are:

1. Hybrid;
2. interaction pathway-only;
3. evidence pathway-only.

The mean represents average test performance across the five independently trained folds.

The standard deviation describes how much performance varied depending on which participants formed the held-out test partition.

The principal cross-validation result will be reported in the form:

$$
\text{mean} \pm \text{standard deviation}.
$$

These fold-level summaries are kept separate from the pooled out-of-fold metrics that will be calculated later. The two forms of evaluation answer related but different questions:

- fold mean and standard deviation describe stability across splits;
- pooled metrics describe performance across all 544 out-of-fold predictions together.

In [ ]:
# ============================================================
# 3. Calculating fold-level cross-validation statistics
# ============================================================

# ------------------------------------------------------------
# Metric columns included in the fold summary
# ------------------------------------------------------------

PERFORMANCE_METRICS = [
    "ROC_AUC",
    "AVERAGE_PRECISION",
    "ACCURACY",
    "BALANCED_ACCURACY",
    "SENSITIVITY",
    "SPECIFICITY",
    "PRECISION",
    "F1",
    "BRIER_SCORE",
    "NEGATIVE_LOG_LIKELIHOOD",
    "EXPECTED_CALIBRATION_ERROR",
    "MEAN_UNCERTAINTY",
]


# ------------------------------------------------------------
# Confirm that all required metric columns are available
# ------------------------------------------------------------

missing_metric_columns = [
    metric
    for metric in PERFORMANCE_METRICS
    if metric not in all_fold_metric_summaries.columns
]


if missing_metric_columns:

    raise KeyError(
        "The following required metric columns are missing:\n"
        + "\n".join(
            missing_metric_columns
        )
    )


# ------------------------------------------------------------
# Preserve a clear model-output order
# ------------------------------------------------------------

OUTPUT_ORDER = [
    "Hybrid",
    "3MT-only",
    "TMC-only",
]


all_fold_metric_summaries[
    "OUTPUT"
] = pd.Categorical(
    all_fold_metric_summaries[
        "OUTPUT"
    ],
    categories=OUTPUT_ORDER,
    ordered=True,
)


all_fold_metric_summaries = (
    all_fold_metric_summaries
    .sort_values(
        [
            "OUTPUT",
            "FOLD",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Long-format summary:
# one row per output and metric
# ------------------------------------------------------------

summary_rows = []

for output_name in OUTPUT_ORDER:

    output_rows = (
        all_fold_metric_summaries.loc[
            all_fold_metric_summaries[
                "OUTPUT"
            ]
            == output_name
        ]
    )


    if len(
        output_rows
    ) != len(
        OUTER_FOLDS
    ):

        raise ValueError(
            f"{output_name} does not contain exactly "
            f"{len(OUTER_FOLDS)} fold rows."
        )


    for metric in PERFORMANCE_METRICS:

        metric_values = (
            pd.to_numeric(
                output_rows[
                    metric
                ],
                errors="raise",
            )
            .to_numpy(
                dtype=float
            )
        )


        summary_rows.append(
            {
                "OUTPUT":
                    output_name,

                "METRIC":
                    metric,

                "FOLD_COUNT":
                    int(
                        len(
                            metric_values
                        )
                    ),

                "MEAN":
                    float(
                        np.mean(
                            metric_values
                        )
                    ),

                # I use the sample standard deviation because
                # the five folds are treated as observed samples
                # of split-dependent model performance.
                "STANDARD_DEVIATION":
                    float(
                        np.std(
                            metric_values,
                            ddof=1,
                        )
                    ),

                "MINIMUM":
                    float(
                        np.min(
                            metric_values
                        )
                    ),

                "MAXIMUM":
                    float(
                        np.max(
                            metric_values
                        )
                    ),
            }
        )


cross_validation_summary_long = pd.DataFrame(
    summary_rows
)


# ------------------------------------------------------------
# Add a thesis-friendly mean ± standard-deviation column
# ------------------------------------------------------------

cross_validation_summary_long[
    "MEAN_PLUS_MINUS_SD"
] = (
    cross_validation_summary_long[
        "MEAN"
    ]
    .map(
        lambda value:
            f"{value:.4f}"
    )
    + " ± "
    + cross_validation_summary_long[
        "STANDARD_DEVIATION"
    ]
    .map(
        lambda value:
            f"{value:.4f}"
    )
)


# ------------------------------------------------------------
# Wide thesis table:
# one row per model output
# ------------------------------------------------------------

cross_validation_summary_wide = (
    cross_validation_summary_long
    .pivot(
        index="OUTPUT",
        columns="METRIC",
        values="MEAN_PLUS_MINUS_SD",
    )
    .reset_index()
)


# I restore the intended output order after pivoting.
cross_validation_summary_wide[
    "OUTPUT"
] = pd.Categorical(
    cross_validation_summary_wide[
        "OUTPUT"
    ],
    categories=OUTPUT_ORDER,
    ordered=True,
)


cross_validation_summary_wide = (
    cross_validation_summary_wide
    .sort_values(
        "OUTPUT"
    )
    .reset_index(
        drop=True
    )
)


# I place the principal clinical-performance metrics first.
ordered_wide_columns = [
    "OUTPUT",
    "ROC_AUC",
    "AVERAGE_PRECISION",
    "BALANCED_ACCURACY",
    "SENSITIVITY",
    "SPECIFICITY",
    "ACCURACY",
    "PRECISION",
    "F1",
    "BRIER_SCORE",
    "NEGATIVE_LOG_LIKELIHOOD",
    "EXPECTED_CALIBRATION_ERROR",
    "MEAN_UNCERTAINTY",
]


cross_validation_summary_wide = (
    cross_validation_summary_wide[
        ordered_wide_columns
    ]
)


# ------------------------------------------------------------
# Fold-by-fold principal performance table
# ------------------------------------------------------------

fold_level_principal_metrics = (
    all_fold_metric_summaries[
        [
            "FOLD",
            "OUTPUT",
            "ROC_AUC",
            "AVERAGE_PRECISION",
            "BALANCED_ACCURACY",
            "SENSITIVITY",
            "SPECIFICITY",
            "BRIER_SCORE",
            "EXPECTED_CALIBRATION_ERROR",
            "MEAN_UNCERTAINTY",
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# Save all summary tables
# ------------------------------------------------------------

CV_SUMMARY_LONG_PATH = (
    TABLE_DIR
    / "cross_validation_summary_long.csv"
)

CV_SUMMARY_WIDE_PATH = (
    TABLE_DIR
    / "cross_validation_summary_mean_sd.csv"
)

FOLD_LEVEL_PRINCIPAL_METRICS_PATH = (
    TABLE_DIR
    / "fold_level_principal_test_metrics.csv"
)


cross_validation_summary_long.to_csv(
    CV_SUMMARY_LONG_PATH,
    index=False,
)

cross_validation_summary_wide.to_csv(
    CV_SUMMARY_WIDE_PATH,
    index=False,
)

fold_level_principal_metrics.to_csv(
    FOLD_LEVEL_PRINCIPAL_METRICS_PATH,
    index=False,
)


# ------------------------------------------------------------
# Display fold-level results
# ------------------------------------------------------------

print("=" * 72)
print("FOLD-LEVEL CROSS-VALIDATION SUMMARY")
print("=" * 72)


print(
    "\nFold-by-fold principal test metrics:"
)

display(
    fold_level_principal_metrics.round(
        6
    )
)


print(
    "\nCross-validation mean ± sample standard deviation:"
)

display(
    cross_validation_summary_wide
)


# ------------------------------------------------------------
# Display the principal ROC AUC comparison explicitly
# ------------------------------------------------------------

roc_auc_summary = (
    cross_validation_summary_long.loc[
        cross_validation_summary_long[
            "METRIC"
        ]
        == "ROC_AUC",
        [
            "OUTPUT",
            "MEAN",
            "STANDARD_DEVIATION",
            "MINIMUM",
            "MAXIMUM",
            "MEAN_PLUS_MINUS_SD",
        ],
    ]
    .reset_index(
        drop=True
    )
)


print(
    "\nROC AUC summary:"
)

display(
    roc_auc_summary.round(
        {
            "MEAN":
                6,

            "STANDARD_DEVIATION":
                6,

            "MINIMUM":
                6,

            "MAXIMUM":
                6,
        }
    )
)


print(
    "\nLong-format cross-validation summary saved to:\n"
    f"{CV_SUMMARY_LONG_PATH}"
)

print(
    "\nMean ± standard-deviation table saved to:\n"
    f"{CV_SUMMARY_WIDE_PATH}"
)

print(
    "\nFold-level principal metrics saved to:\n"
    f"{FOLD_LEVEL_PRINCIPAL_METRICS_PATH}"
)

## 1.3. Inspecting the pooled out-of-fold prediction schema

Before calculating pooled performance, The notebook inspects the exact columns saved in the participant-level prediction files.

This ensures that the pooled analysis uses the authoritative output names produced by the training notebooks.

I specifically need to identify:

- the participant identifier;
- the true binary target;
- the predicted pMCI probability for each pathway;
- the predicted class for each pathway;
- the uncertainty for each pathway;
- the interaction pathway and evidence pathway reliability weights;
- modality-count and modality-availability fields.

No metric is calculated in this step. confirm the structure and data types of the pooled prediction table.

In [ ]:
# ============================================================
# 4. Inspecting the pooled out-of-fold prediction schema
# ============================================================

# ------------------------------------------------------------
# Display the complete column inventory
# ------------------------------------------------------------

prediction_schema = pd.DataFrame(
    {
        "COLUMN_POSITION":
            np.arange(
                len(
                    pooled_out_of_fold_predictions.columns
                )
            ),

        "COLUMN_NAME":
            pooled_out_of_fold_predictions.columns,

        "DATA_TYPE":
            [
                str(
                    pooled_out_of_fold_predictions[
                        column
                    ].dtype
                )
                for column in (
                    pooled_out_of_fold_predictions.columns
                )
            ],

        "NON_MISSING_COUNT":
            [
                int(
                    pooled_out_of_fold_predictions[
                        column
                    ].notna().sum()
                )
                for column in (
                    pooled_out_of_fold_predictions.columns
                )
            ],

        "UNIQUE_COUNT":
            [
                int(
                    pooled_out_of_fold_predictions[
                        column
                    ].nunique(
                        dropna=True
                    )
                )
                for column in (
                    pooled_out_of_fold_predictions.columns
                )
            ],
    }
)


# ------------------------------------------------------------
# Show representative values for every column
# ------------------------------------------------------------

sample_value_rows = []

for column in pooled_out_of_fold_predictions.columns:

    non_missing_values = (
        pooled_out_of_fold_predictions[
            column
        ]
        .dropna()
        .head(
            5
        )
        .tolist()
    )

    sample_value_rows.append(
        {
            "COLUMN_NAME":
                column,

            "SAMPLE_VALUES":
                non_missing_values,
        }
    )


prediction_sample_values = pd.DataFrame(
    sample_value_rows
)


# ------------------------------------------------------------
# Inspect target balance by fold
# ------------------------------------------------------------

possible_target_columns = [
    "target",
    "TARGET",
    "true_target",
    "TRUE_TARGET",
    "label",
    "LABEL",
]


target_column = next(
    (
        column
        for column in possible_target_columns
        if column in pooled_out_of_fold_predictions.columns
    ),
    None,
)


if target_column is not None:

    target_balance_by_fold = (
        pooled_out_of_fold_predictions
        .groupby(
            [
                "FOLD",
                target_column,
            ],
            dropna=False,
        )
        .size()
        .reset_index(
            name="PARTICIPANTS"
        )
    )

else:

    target_balance_by_fold = pd.DataFrame()


# ------------------------------------------------------------
# Display the schema
# ------------------------------------------------------------

print("=" * 72)
print("POOLED OUT-OF-FOLD PREDICTION SCHEMA")
print("=" * 72)


print(
    "\nPrediction-table dimensions:"
)

print(
    f"  rows: {len(pooled_out_of_fold_predictions)}"
)

print(
    "  columns: "
    f"{len(pooled_out_of_fold_predictions.columns)}"
)


print(
    "\nComplete column schema:"
)

display(
    prediction_schema
)


print(
    "\nRepresentative values:"
)

display(
    prediction_sample_values
)


if target_column is not None:

    print(
        "\nDetected target column: "
        f"{target_column}"
    )

    print(
        "\nTarget balance by fold:"
    )

    display(
        target_balance_by_fold
    )

else:

    print(
        "\nNo target column was selected automatically."
    )

    print(
        "The exact target-column name will be chosen after "
        "reviewing the displayed schema."
    )


# ------------------------------------------------------------
# Save the schema inventory
# ------------------------------------------------------------

PREDICTION_SCHEMA_PATH = (
    TABLE_DIR
    / "pooled_prediction_schema.csv"
)

PREDICTION_SAMPLE_VALUES_PATH = (
    TABLE_DIR
    / "pooled_prediction_sample_values.csv"
)


prediction_schema.to_csv(
    PREDICTION_SCHEMA_PATH,
    index=False,
)

prediction_sample_values.to_csv(
    PREDICTION_SAMPLE_VALUES_PATH,
    index=False,
)


print(
    "\nPrediction schema saved to:\n"
    f"{PREDICTION_SCHEMA_PATH}"
)

print(
    "\nRepresentative values saved to:\n"
    f"{PREDICTION_SAMPLE_VALUES_PATH}"
)

## 1.4. Calculating pooled out-of-fold performance

calculate performance using all 544 out-of-fold predictions together.

Each participant contributes exactly one prediction made by a model that did not use that participant for training or validation.

The notebook calculates pooled metrics separately for:

1. the final Hybrid output;
2. the interaction pathway-only pathway;
3. the evidence pathway-only pathway.

The pooled metrics include:

- ROC AUC;
- average precision;
- accuracy;
- balanced accuracy;
- sensitivity;
- specificity;
- precision;
- F1 score;
- Brier score;
- negative log-likelihood;
- expected calibration error;
- mean uncertainty.

For threshold-dependent metrics, I retain the fixed threshold of \(0.50\).

The pooled result differs slightly from the simple average across folds because it operates on all participant-level predictions together. It therefore weights folds according to their number of participants and calculates ranking metrics across the entire cohort rather than within each fold separately.

In [ ]:
# ============================================================
# 5. Calculating pooled out-of-fold performance
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    roc_auc_score,
)


# ------------------------------------------------------------
# Authoritative pooled target
# ------------------------------------------------------------

pooled_targets = (
    pooled_out_of_fold_predictions[
        "TARGET"
    ]
    .to_numpy(
        dtype=int
    )
)


# ------------------------------------------------------------
# Probability and uncertainty columns for each output
# ------------------------------------------------------------

POOLED_OUTPUT_COLUMNS = {
    "Hybrid": {
        "probability":
            "FINAL_P_pMCI",

        "uncertainty":
            "FINAL_UNCERTAINTY",
    },

    "3MT-only": {
        "probability":
            "THREE_MT_P_pMCI",

        "uncertainty":
            "THREE_MT_UNCERTAINTY",
    },

    "TMC-only": {
        "probability":
            "TMC_P_pMCI",

        "uncertainty":
            "TMC_UNCERTAINTY",
    },
}


CLASSIFICATION_THRESHOLD = 0.50
CALIBRATION_BIN_COUNT = 10


# ------------------------------------------------------------
# Expected calibration error
# ------------------------------------------------------------

def calculate_expected_calibration_error(
    targets,
    probabilities,
    number_of_bins=10,
):
    """
    Calculate expected calibration error using equal-width
    probability bins between zero and one.
    """

    bin_edges = np.linspace(
        0.0,
        1.0,
        number_of_bins + 1,
    )

    expected_calibration_error = 0.0

    sample_count = len(
        targets
    )


    for bin_index in range(
        number_of_bins
    ):

        lower_edge = bin_edges[
            bin_index
        ]

        upper_edge = bin_edges[
            bin_index + 1
        ]


        # I include the upper edge only for the final bin so
        # that a probability of exactly 1.0 is not excluded.
        if bin_index == (
            number_of_bins - 1
        ):

            in_bin = (
                (probabilities >= lower_edge)
                & (probabilities <= upper_edge)
            )

        else:

            in_bin = (
                (probabilities >= lower_edge)
                & (probabilities < upper_edge)
            )


        bin_count = int(
            np.sum(
                in_bin
            )
        )


        if bin_count == 0:
            continue


        bin_accuracy = float(
            np.mean(
                targets[
                    in_bin
                ]
            )
        )

        bin_confidence = float(
            np.mean(
                probabilities[
                    in_bin
                ]
            )
        )


        expected_calibration_error += (
            bin_count
            / sample_count
        ) * abs(
            bin_accuracy
            - bin_confidence
        )


    return float(
        expected_calibration_error
    )


# ------------------------------------------------------------
# Metric calculation for one output
# ------------------------------------------------------------

def calculate_pooled_binary_metrics(
    targets,
    probabilities,
    uncertainties,
    threshold=0.50,
):
    """
    Calculate pooled binary-classification and calibration
    metrics for one model output.
    """

    predicted_classes = (
        probabilities
        >= threshold
    ).astype(
        int
    )


    confusion = confusion_matrix(
        targets,
        predicted_classes,
        labels=[
            0,
            1,
        ],
    )


    true_negative = int(
        confusion[
            0,
            0,
        ]
    )

    false_positive = int(
        confusion[
            0,
            1,
        ]
    )

    false_negative = int(
        confusion[
            1,
            0,
        ]
    )

    true_positive = int(
        confusion[
            1,
            1,
        ]
    )


    sensitivity = (
        true_positive
        / (
            true_positive
            + false_negative
        )
    )

    specificity = (
        true_negative
        / (
            true_negative
            + false_positive
        )
    )


    clipped_probabilities = np.clip(
        probabilities,
        1e-7,
        1.0 - 1e-7,
    )


    return {
        "ROC_AUC":
            float(
                roc_auc_score(
                    targets,
                    probabilities,
                )
            ),

        "AVERAGE_PRECISION":
            float(
                average_precision_score(
                    targets,
                    probabilities,
                )
            ),

        "ACCURACY":
            float(
                accuracy_score(
                    targets,
                    predicted_classes,
                )
            ),

        "BALANCED_ACCURACY":
            float(
                balanced_accuracy_score(
                    targets,
                    predicted_classes,
                )
            ),

        "SENSITIVITY":
            float(
                sensitivity
            ),

        "SPECIFICITY":
            float(
                specificity
            ),

        "PRECISION":
            float(
                precision_score(
                    targets,
                    predicted_classes,
                    zero_division=0,
                )
            ),

        "F1":
            float(
                f1_score(
                    targets,
                    predicted_classes,
                    zero_division=0,
                )
            ),

        "BRIER_SCORE":
            float(
                brier_score_loss(
                    targets,
                    probabilities,
                )
            ),

        "NEGATIVE_LOG_LIKELIHOOD":
            float(
                log_loss(
                    targets,
                    np.column_stack(
                        [
                            1.0
                            - clipped_probabilities,

                            clipped_probabilities,
                        ]
                    ),
                    labels=[
                        0,
                        1,
                    ],
                )
            ),

        "EXPECTED_CALIBRATION_ERROR":
            calculate_expected_calibration_error(
                targets=targets,
                probabilities=probabilities,
                number_of_bins=
                    CALIBRATION_BIN_COUNT,
            ),

        "MEAN_UNCERTAINTY":
            float(
                np.mean(
                    uncertainties
                )
            ),

        "STANDARD_DEVIATION_UNCERTAINTY":
            float(
                np.std(
                    uncertainties,
                    ddof=1,
                )
            ),

        "TRUE_NEGATIVE":
            true_negative,

        "FALSE_POSITIVE":
            false_positive,

        "FALSE_NEGATIVE":
            false_negative,

        "TRUE_POSITIVE":
            true_positive,

        "PREDICTED_POSITIVE":
            int(
                np.sum(
                    predicted_classes
                    == 1
                )
            ),

        "PREDICTED_NEGATIVE":
            int(
                np.sum(
                    predicted_classes
                    == 0
                )
            ),
    }


# ------------------------------------------------------------
# Calculate pooled metrics for all three outputs
# ------------------------------------------------------------

pooled_metric_rows = []

pooled_prediction_columns = (
    pooled_out_of_fold_predictions.copy()
)


for output_name in OUTPUT_ORDER:

    probability_column = (
        POOLED_OUTPUT_COLUMNS[
            output_name
        ][
            "probability"
        ]
    )

    uncertainty_column = (
        POOLED_OUTPUT_COLUMNS[
            output_name
        ][
            "uncertainty"
        ]
    )


    probabilities = (
        pooled_out_of_fold_predictions[
            probability_column
        ]
        .to_numpy(
            dtype=float
        )
    )

    uncertainties = (
        pooled_out_of_fold_predictions[
            uncertainty_column
        ]
        .to_numpy(
            dtype=float
        )
    )


    if np.any(
        ~np.isfinite(
            probabilities
        )
    ):

        raise FloatingPointError(
            f"{output_name} contains non-finite probabilities."
        )


    if np.any(
        (
            probabilities
            < 0.0
        )
        |
        (
            probabilities
            > 1.0
        )
    ):

        raise ValueError(
            f"{output_name} contains probabilities outside [0, 1]."
        )


    if np.any(
        ~np.isfinite(
            uncertainties
        )
    ):

        raise FloatingPointError(
            f"{output_name} contains non-finite uncertainties."
        )


    output_metrics = (
        calculate_pooled_binary_metrics(
            targets=pooled_targets,
            probabilities=probabilities,
            uncertainties=uncertainties,
            threshold=
                CLASSIFICATION_THRESHOLD,
        )
    )


    output_metrics[
        "OUTPUT"
    ] = output_name

    output_metrics[
        "PARTICIPANTS"
    ] = int(
        len(
            pooled_targets
        )
    )


    pooled_metric_rows.append(
        output_metrics
    )


    # I save thresholded classes for later participant-level
    # error and uncertainty analyses.
    safe_output_name = (
        output_name
        .upper()
        .replace(
            "-",
            "_",
        )
    )

    pooled_prediction_columns[
        f"{safe_output_name}_POOLED_PREDICTED_CLASS"
    ] = (
        probabilities
        >= CLASSIFICATION_THRESHOLD
    ).astype(
        int
    )


# ------------------------------------------------------------
# Final pooled metric table
# ------------------------------------------------------------

pooled_performance_summary = pd.DataFrame(
    pooled_metric_rows
)


pooled_performance_summary[
    "OUTPUT"
] = pd.Categorical(
    pooled_performance_summary[
        "OUTPUT"
    ],
    categories=OUTPUT_ORDER,
    ordered=True,
)


pooled_performance_summary = (
    pooled_performance_summary
    .sort_values(
        "OUTPUT"
    )
    .reset_index(
        drop=True
    )
)


ordered_pooled_columns = [
    "OUTPUT",
    "PARTICIPANTS",
    "ROC_AUC",
    "AVERAGE_PRECISION",
    "ACCURACY",
    "BALANCED_ACCURACY",
    "SENSITIVITY",
    "SPECIFICITY",
    "PRECISION",
    "F1",
    "BRIER_SCORE",
    "NEGATIVE_LOG_LIKELIHOOD",
    "EXPECTED_CALIBRATION_ERROR",
    "MEAN_UNCERTAINTY",
    "STANDARD_DEVIATION_UNCERTAINTY",
    "TRUE_NEGATIVE",
    "FALSE_POSITIVE",
    "FALSE_NEGATIVE",
    "TRUE_POSITIVE",
    "PREDICTED_NEGATIVE",
    "PREDICTED_POSITIVE",
]


pooled_performance_summary = (
    pooled_performance_summary[
        ordered_pooled_columns
    ]
)


# ------------------------------------------------------------
# Add hybrid gate statistics
# ------------------------------------------------------------

hybrid_gate_summary = pd.DataFrame(
    {
        "STATISTIC": [
            "mean_3MT_weight",
            "standard_deviation_3MT_weight",
            "minimum_3MT_weight",
            "maximum_3MT_weight",
            "mean_TMC_weight",
            "standard_deviation_TMC_weight",
            "minimum_TMC_weight",
            "maximum_TMC_weight",
            "maximum_weight_sum_error",
        ],

        "VALUE": [
            float(
                pooled_out_of_fold_predictions[
                    "W_3MT"
                ].mean()
            ),

            float(
                pooled_out_of_fold_predictions[
                    "W_3MT"
                ].std(
                    ddof=1
                )
            ),

            float(
                pooled_out_of_fold_predictions[
                    "W_3MT"
                ].min()
            ),

            float(
                pooled_out_of_fold_predictions[
                    "W_3MT"
                ].max()
            ),

            float(
                pooled_out_of_fold_predictions[
                    "W_TMC"
                ].mean()
            ),

            float(
                pooled_out_of_fold_predictions[
                    "W_TMC"
                ].std(
                    ddof=1
                )
            ),

            float(
                pooled_out_of_fold_predictions[
                    "W_TMC"
                ].min()
            ),

            float(
                pooled_out_of_fold_predictions[
                    "W_TMC"
                ].max()
            ),

            float(
                np.max(
                    np.abs(
                        (
                            pooled_out_of_fold_predictions[
                                "W_3MT"
                            ]
                            +
                            pooled_out_of_fold_predictions[
                                "W_TMC"
                            ]
                        )
                        - 1.0
                    )
                )
            ),
        ],
    }
)


# ------------------------------------------------------------
# Save pooled outputs
# ------------------------------------------------------------

POOLED_PERFORMANCE_PATH = (
    TABLE_DIR
    / "pooled_out_of_fold_performance.csv"
)

POOLED_GATE_SUMMARY_PATH = (
    TABLE_DIR
    / "pooled_hybrid_gate_summary.csv"
)

POOLED_ENRICHED_PREDICTIONS_PATH = (
    POOLED_PREDICTION_DIR
    / "pooled_out_of_fold_predictions_with_classes.csv"
)


pooled_performance_summary.to_csv(
    POOLED_PERFORMANCE_PATH,
    index=False,
)

hybrid_gate_summary.to_csv(
    POOLED_GATE_SUMMARY_PATH,
    index=False,
)

pooled_prediction_columns.to_csv(
    POOLED_ENRICHED_PREDICTIONS_PATH,
    index=False,
)


# ------------------------------------------------------------
# Display pooled results
# ------------------------------------------------------------

print("=" * 72)
print("POOLED OUT-OF-FOLD PERFORMANCE")
print("=" * 72)


print(
    "\nPooled class balance:"
)

print(
    "  sMCI: "
    f"{int(np.sum(pooled_targets == 0))}"
)

print(
    "  pMCI: "
    f"{int(np.sum(pooled_targets == 1))}"
)

print(
    "  total: "
    f"{len(pooled_targets)}"
)


# I show only the principal metrics in the notebook output.
pooled_performance_display = (
    pooled_performance_summary[
        [
            "OUTPUT",
            "ROC_AUC",
            "AVERAGE_PRECISION",
            "BALANCED_ACCURACY",
            "SENSITIVITY",
            "SPECIFICITY",
            "BRIER_SCORE",
            "EXPECTED_CALIBRATION_ERROR",
            "MEAN_UNCERTAINTY",
        ]
    ]
    .round(
        6
    )
)

display(
    pooled_performance_display
)


print(
    "\nHybrid gate summary:"
)

display(
    hybrid_gate_summary.round(
        8
    )
)


print(
    "\nPooled performance saved to:\n"
    f"{POOLED_PERFORMANCE_PATH}"
)

print(
    "\nPooled gate summary saved to:\n"
    f"{POOLED_GATE_SUMMARY_PATH}"
)

print(
    "\nEnriched pooled predictions saved to:\n"
    f"{POOLED_ENRICHED_PREDICTIONS_PATH}"
)

## 1.5. Pooled out-of-fold performance overview

The table generated above contains the final pooled results across all 544 held-out participants. The Hybrid ROC AUC in this table is the principal ROC AUC for the pre-baseline temporal-sensitivity experiment.

The interaction pathway-only and evidence pathway-only rows are retained because they are outputs of the same trained architecture and help show how the fixed Hybrid combination relates to its two constituent pathways.


## 1.6. Confirming the fixed 50/50 fusion weights

This experiment removes the learned participant-specific reliability gate. The saved participant-level predictions should therefore contain a interaction pathway weight of exactly 0.50 and a evidence pathway weight of exactly 0.50 for every participant in every fold.


In [ ]:
# ============================================================
# 6. Confirming the fixed 50/50 fusion weights
# ============================================================

# ------------------------------------------------------------
# Summarise the saved pathway weights by fold
# ------------------------------------------------------------

fixed_weight_summary_by_fold = (
    pooled_out_of_fold_predictions
    .groupby(
        "FOLD"
    )
    .agg(
        PARTICIPANTS=(
            "RID",
            "size",
        ),

        MEAN_3MT_WEIGHT=(
            "W_3MT",
            "mean",
        ),

        MINIMUM_3MT_WEIGHT=(
            "W_3MT",
            "min",
        ),

        MAXIMUM_3MT_WEIGHT=(
            "W_3MT",
            "max",
        ),

        MEAN_TMC_WEIGHT=(
            "W_TMC",
            "mean",
        ),

        MINIMUM_TMC_WEIGHT=(
            "W_TMC",
            "min",
        ),

        MAXIMUM_TMC_WEIGHT=(
            "W_TMC",
            "max",
        ),
    )
    .reset_index()
)


# ------------------------------------------------------------
# Confirm the intended fixed values
# ------------------------------------------------------------

maximum_three_mt_weight_error = float(
    np.max(
        np.abs(
            pooled_out_of_fold_predictions[
                "W_3MT"
            ].to_numpy(
                dtype=float
            )
            - 0.5
        )
    )
)

maximum_tmc_weight_error = float(
    np.max(
        np.abs(
            pooled_out_of_fold_predictions[
                "W_TMC"
            ].to_numpy(
                dtype=float
            )
            - 0.5
        )
    )
)

maximum_weight_sum_error = float(
    np.max(
        np.abs(
            (
                pooled_out_of_fold_predictions[
                    "W_3MT"
                ].to_numpy(
                    dtype=float
                )
                +
                pooled_out_of_fold_predictions[
                    "W_TMC"
                ].to_numpy(
                    dtype=float
                )
            )
            - 1.0
        )
    )
)


if maximum_three_mt_weight_error > 1e-8:
    raise ValueError(
        "The saved 3MT weights are not fixed at 0.50."
    )

if maximum_tmc_weight_error > 1e-8:
    raise ValueError(
        "The saved TMC weights are not fixed at 0.50."
    )

if maximum_weight_sum_error > 1e-8:
    raise ValueError(
        "The fixed 3MT and TMC weights do not sum to one."
    )


# ------------------------------------------------------------
# Save and display the validation table
# ------------------------------------------------------------

FIXED_WEIGHT_SUMMARY_PATH = (
    TABLE_DIR
    / "fixed_equal_fusion_weight_summary_by_fold.csv"
)

fixed_weight_summary_by_fold.to_csv(
    FIXED_WEIGHT_SUMMARY_PATH,
    index=False,
)


print("=" * 72)
print("FIXED 50/50 FUSION WEIGHT VALIDATION")
print("=" * 72)

display(
    fixed_weight_summary_by_fold.round(
        8
    )
)

print(
    "\nMaximum 3MT-weight deviation from 0.50:"
)

print(
    f"  {maximum_three_mt_weight_error:.10f}"
)

print(
    "\nMaximum TMC-weight deviation from 0.50:"
)

print(
    f"  {maximum_tmc_weight_error:.10f}"
)

print(
    "\nMaximum error in W_3MT + W_TMC = 1:"
)

print(
    f"  {maximum_weight_sum_error:.10f}"
)

print(
    "\nFixed-weight summary saved to:\n"
    f"{FIXED_WEIGHT_SUMMARY_PATH}"
)


### 1.6.1. Fixed-fusion verification for the temporal experiment

The table above verifies that the final Hybrid output used the intended fixed 0.50/0.50 pathway weighting in every fold. The two pathway evidence scales may still differ because they remain trainable; only the participant-specific weighting mechanism has been removed.


## 1.7. Analysing whether uncertainty identifies prediction errors

The model is designed not only to predict MCI progression, but also to express uncertainty about its predictions.

A useful uncertainty estimate should generally be higher for difficult or incorrect cases than for correct cases.

I therefore compare uncertainty for:

- correct predictions;
- incorrect predictions;
- true negatives;
- false positives;
- false negatives;
- true positives.

I perform this analysis separately for:

1. the Hybrid output;
2. the interaction pathway-only pathway;
3. the evidence pathway-only pathway.

I also calculate the correlation between uncertainty and absolute prediction error.

For each participant, absolute prediction error is defined as:

$$
\left| y - \hat{p} \right|,
$$

where \(y\) is the true binary target and \(\hat{p}\) is the predicted probability of pMCI.

A positive correlation would indicate that uncertainty tends to increase as prediction error becomes larger.

In [ ]:
# ============================================================
# 7. Analysing uncertainty for correct and incorrect predictions
# ============================================================

from scipy.stats import (
    mannwhitneyu,
    spearmanr,
)


# ------------------------------------------------------------
# Participant-level uncertainty analysis
# ------------------------------------------------------------

uncertainty_analysis_rows = []
uncertainty_error_correlation_rows = []


for output_name in OUTPUT_ORDER:

    probability_column = (
        POOLED_OUTPUT_COLUMNS[
            output_name
        ][
            "probability"
        ]
    )

    uncertainty_column = (
        POOLED_OUTPUT_COLUMNS[
            output_name
        ][
            "uncertainty"
        ]
    )


    probabilities = (
        pooled_out_of_fold_predictions[
            probability_column
        ]
        .to_numpy(
            dtype=float
        )
    )

    uncertainties = (
        pooled_out_of_fold_predictions[
            uncertainty_column
        ]
        .to_numpy(
            dtype=float
        )
    )

    predicted_classes = (
        probabilities
        >= CLASSIFICATION_THRESHOLD
    ).astype(
        int
    )

    correct_predictions = (
        predicted_classes
        == pooled_targets
    )

    absolute_probability_error = np.abs(
        pooled_targets
        - probabilities
    )


    # --------------------------------------------------------
    # Compare uncertainty for correct and incorrect cases
    # --------------------------------------------------------

    correct_uncertainty = uncertainties[
        correct_predictions
    ]

    incorrect_uncertainty = uncertainties[
        ~correct_predictions
    ]


    if (
        len(
            correct_uncertainty
        )
        > 0
        and len(
            incorrect_uncertainty
        )
        > 0
    ):

        mann_whitney_result = mannwhitneyu(
            incorrect_uncertainty,
            correct_uncertainty,
            alternative="greater",
        )

        mann_whitney_u = float(
            mann_whitney_result.statistic
        )

        mann_whitney_p = float(
            mann_whitney_result.pvalue
        )

    else:

        mann_whitney_u = np.nan
        mann_whitney_p = np.nan


    uncertainty_analysis_rows.append(
        {
            "OUTPUT":
                output_name,

            "GROUP":
                "Correct",

            "PARTICIPANTS":
                int(
                    len(
                        correct_uncertainty
                    )
                ),

            "MEAN_UNCERTAINTY":
                float(
                    np.mean(
                        correct_uncertainty
                    )
                ),

            "MEDIAN_UNCERTAINTY":
                float(
                    np.median(
                        correct_uncertainty
                    )
                ),

            "STANDARD_DEVIATION":
                float(
                    np.std(
                        correct_uncertainty,
                        ddof=1,
                    )
                ),
        }
    )


    uncertainty_analysis_rows.append(
        {
            "OUTPUT":
                output_name,

            "GROUP":
                "Incorrect",

            "PARTICIPANTS":
                int(
                    len(
                        incorrect_uncertainty
                    )
                ),

            "MEAN_UNCERTAINTY":
                float(
                    np.mean(
                        incorrect_uncertainty
                    )
                ),

            "MEDIAN_UNCERTAINTY":
                float(
                    np.median(
                        incorrect_uncertainty
                    )
                ),

            "STANDARD_DEVIATION":
                float(
                    np.std(
                        incorrect_uncertainty,
                        ddof=1,
                    )
                ),
        }
    )


    # --------------------------------------------------------
    # Compare uncertainty across confusion-matrix groups
    # --------------------------------------------------------

    confusion_groups = {
        "True negative":
            (
                (pooled_targets == 0)
                & (predicted_classes == 0)
            ),

        "False positive":
            (
                (pooled_targets == 0)
                & (predicted_classes == 1)
            ),

        "False negative":
            (
                (pooled_targets == 1)
                & (predicted_classes == 0)
            ),

        "True positive":
            (
                (pooled_targets == 1)
                & (predicted_classes == 1)
            ),
    }


    for group_name, group_mask in (
        confusion_groups.items()
    ):

        group_uncertainty = uncertainties[
            group_mask
        ]

        uncertainty_analysis_rows.append(
            {
                "OUTPUT":
                    output_name,

                "GROUP":
                    group_name,

                "PARTICIPANTS":
                    int(
                        len(
                            group_uncertainty
                        )
                    ),

                "MEAN_UNCERTAINTY":
                    float(
                        np.mean(
                            group_uncertainty
                        )
                    ),

                "MEDIAN_UNCERTAINTY":
                    float(
                        np.median(
                            group_uncertainty
                        )
                    ),

                "STANDARD_DEVIATION":
                    float(
                        np.std(
                            group_uncertainty,
                            ddof=1,
                        )
                    )
                    if len(
                        group_uncertainty
                    ) > 1
                    else np.nan,
            }
        )


    # --------------------------------------------------------
    # Correlation between uncertainty and prediction error
    # --------------------------------------------------------

    spearman_result = spearmanr(
        uncertainties,
        absolute_probability_error,
    )


    uncertainty_error_correlation_rows.append(
        {
            "OUTPUT":
                output_name,

            "SPEARMAN_CORRELATION":
                float(
                    spearman_result.statistic
                ),

            "P_VALUE":
                float(
                    spearman_result.pvalue
                ),

            "MEAN_CORRECT_UNCERTAINTY":
                float(
                    np.mean(
                        correct_uncertainty
                    )
                ),

            "MEAN_INCORRECT_UNCERTAINTY":
                float(
                    np.mean(
                        incorrect_uncertainty
                    )
                ),

            "INCORRECT_MINUS_CORRECT":
                float(
                    np.mean(
                        incorrect_uncertainty
                    )
                    -
                    np.mean(
                        correct_uncertainty
                    )
                ),

            "MANN_WHITNEY_U":
                mann_whitney_u,

            "MANN_WHITNEY_ONE_SIDED_P":
                mann_whitney_p,
        }
    )


# ------------------------------------------------------------
# Final uncertainty-analysis tables
# ------------------------------------------------------------

uncertainty_group_summary = pd.DataFrame(
    uncertainty_analysis_rows
)

uncertainty_error_correlation = pd.DataFrame(
    uncertainty_error_correlation_rows
)


uncertainty_group_summary[
    "OUTPUT"
] = pd.Categorical(
    uncertainty_group_summary[
        "OUTPUT"
    ],
    categories=OUTPUT_ORDER,
    ordered=True,
)


uncertainty_error_correlation[
    "OUTPUT"
] = pd.Categorical(
    uncertainty_error_correlation[
        "OUTPUT"
    ],
    categories=OUTPUT_ORDER,
    ordered=True,
)


uncertainty_group_summary = (
    uncertainty_group_summary
    .sort_values(
        [
            "OUTPUT",
            "GROUP",
        ]
    )
    .reset_index(
        drop=True
    )
)


uncertainty_error_correlation = (
    uncertainty_error_correlation
    .sort_values(
        "OUTPUT"
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Save uncertainty-analysis outputs
# ------------------------------------------------------------

UNCERTAINTY_GROUP_SUMMARY_PATH = (
    TABLE_DIR
    / "uncertainty_by_prediction_group.csv"
)

UNCERTAINTY_ERROR_CORRELATION_PATH = (
    STATISTICS_DIR
    / "uncertainty_prediction_error_association.csv"
)


uncertainty_group_summary.to_csv(
    UNCERTAINTY_GROUP_SUMMARY_PATH,
    index=False,
)

uncertainty_error_correlation.to_csv(
    UNCERTAINTY_ERROR_CORRELATION_PATH,
    index=False,
)


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("=" * 72)
print("UNCERTAINTY AND PREDICTION-ERROR ANALYSIS")
print("=" * 72)


print(
    "\nCorrect versus incorrect prediction uncertainty:"
)

display(
    uncertainty_group_summary.loc[
        uncertainty_group_summary[
            "GROUP"
        ].isin(
            [
                "Correct",
                "Incorrect",
            ]
        )
    ][
        [
            "OUTPUT",
            "GROUP",
            "PARTICIPANTS",
            "MEAN_UNCERTAINTY",
            "MEDIAN_UNCERTAINTY",
            "STANDARD_DEVIATION",
        ]
    ]
    .round(
        6
    )
)


print(
    "\nUncertainty by confusion-matrix outcome:"
)

display(
    uncertainty_group_summary.loc[
        uncertainty_group_summary[
            "GROUP"
        ].isin(
            [
                "True negative",
                "False positive",
                "False negative",
                "True positive",
            ]
        )
    ][
        [
            "OUTPUT",
            "GROUP",
            "PARTICIPANTS",
            "MEAN_UNCERTAINTY",
            "MEDIAN_UNCERTAINTY",
        ]
    ]
    .round(
        6
    )
)


print(
    "\nAssociation between uncertainty and prediction error:"
)

display(
    uncertainty_error_correlation[
        [
            "OUTPUT",
            "SPEARMAN_CORRELATION",
            "P_VALUE",
            "MEAN_CORRECT_UNCERTAINTY",
            "MEAN_INCORRECT_UNCERTAINTY",
            "INCORRECT_MINUS_CORRECT",
            "MANN_WHITNEY_ONE_SIDED_P",
        ]
    ]
    .round(
        6
    )
)


print(
    "\nUncertainty group summary saved to:\n"
    f"{UNCERTAINTY_GROUP_SUMMARY_PATH}"
)

print(
    "\nUncertainty-error association saved to:\n"
    f"{UNCERTAINTY_ERROR_CORRELATION_PATH}"
)

### 1.7.1. Uncertainty and prediction-error analysis

The tables above compare uncertainty for correct and incorrect predictions for each output. A useful uncertainty estimate should generally be higher for errors than for correct predictions.

The current gated results should be interpreted directly rather than reusing conclusions from the earlier experiment.


## 1.8. Analysing performance and uncertainty by modality count

The cohort retains participants even when some modalities are unavailable. The number of available modalities therefore differs across participants.

examine whether predictive performance and uncertainty change according to the number of modalities available at test time.

For each modality-count group, I calculate:

- participant count;
- sMCI and pMCI counts;
- proportion of pMCI participants;
- ROC AUC;
- average precision;
- balanced accuracy;
- sensitivity;
- specificity;
- Brier score;
- mean uncertainty.

The analysis is performed separately for:

1. the Hybrid output;
2. the interaction pathway-only pathway;
3. the evidence pathway-only pathway.

The `ORIGINAL_MODALITY_COUNT` field is used because it represents the modalities genuinely available for each participant. The earlier validation confirmed that `EFFECTIVE_MODALITY_COUNT` was identical during testing, meaning that no additional test-time modality dropout was applied.

Results for small modality-count groups must be interpreted cautiously because their estimates may be unstable.

In [ ]:
# ============================================================
# 8. Analysing performance and uncertainty by modality count
# ============================================================

# ------------------------------------------------------------
# Confirm test-time modality counts remained unchanged
# ------------------------------------------------------------

modality_count_difference = (
    pooled_out_of_fold_predictions[
        "ORIGINAL_MODALITY_COUNT"
    ]
    -
    pooled_out_of_fold_predictions[
        "EFFECTIVE_MODALITY_COUNT"
    ]
)


if not (
    modality_count_difference
    == 0
).all():

    raise ValueError(
        "Original and effective modality counts differ for "
        "at least one test participant."
    )


# ------------------------------------------------------------
# Cohort composition by modality count
# ------------------------------------------------------------

modality_count_distribution = (
    pooled_out_of_fold_predictions
    .groupby(
        "ORIGINAL_MODALITY_COUNT"
    )
    .agg(
        PARTICIPANTS=(
            "RID",
            "size",
        ),

        sMCI_PARTICIPANTS=(
            "TARGET",
            lambda values:
                int(
                    np.sum(
                        values
                        == 0
                    )
                ),
        ),

        pMCI_PARTICIPANTS=(
            "TARGET",
            lambda values:
                int(
                    np.sum(
                        values
                        == 1
                    )
                ),
        ),

        pMCI_PROPORTION=(
            "TARGET",
            "mean",
        ),
    )
    .reset_index()
    .sort_values(
        "ORIGINAL_MODALITY_COUNT"
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Calculate metrics within each modality-count group
# ------------------------------------------------------------

modality_count_metric_rows = []


for modality_count in sorted(
    pooled_out_of_fold_predictions[
        "ORIGINAL_MODALITY_COUNT"
    ].unique()
):

    group_mask = (
        pooled_out_of_fold_predictions[
            "ORIGINAL_MODALITY_COUNT"
        ]
        == modality_count
    )


    group_targets = (
        pooled_out_of_fold_predictions.loc[
            group_mask,
            "TARGET",
        ]
        .to_numpy(
            dtype=int
        )
    )


    for output_name in OUTPUT_ORDER:

        probability_column = (
            POOLED_OUTPUT_COLUMNS[
                output_name
            ][
                "probability"
            ]
        )

        uncertainty_column = (
            POOLED_OUTPUT_COLUMNS[
                output_name
            ][
                "uncertainty"
            ]
        )


        group_probabilities = (
            pooled_out_of_fold_predictions.loc[
                group_mask,
                probability_column,
            ]
            .to_numpy(
                dtype=float
            )
        )

        group_uncertainties = (
            pooled_out_of_fold_predictions.loc[
                group_mask,
                uncertainty_column,
            ]
            .to_numpy(
                dtype=float
            )
        )


        group_predictions = (
            group_probabilities
            >= CLASSIFICATION_THRESHOLD
        ).astype(
            int
        )


        group_confusion_matrix = confusion_matrix(
            group_targets,
            group_predictions,
            labels=[
                0,
                1,
            ],
        )


        true_negative = int(
            group_confusion_matrix[
                0,
                0,
            ]
        )

        false_positive = int(
            group_confusion_matrix[
                0,
                1,
            ]
        )

        false_negative = int(
            group_confusion_matrix[
                1,
                0,
            ]
        )

        true_positive = int(
            group_confusion_matrix[
                1,
                1,
            ]
        )


        negative_count = (
            true_negative
            + false_positive
        )

        positive_count = (
            true_positive
            + false_negative
        )


        sensitivity = (
            true_positive
            / positive_count
            if positive_count > 0
            else np.nan
        )

        specificity = (
            true_negative
            / negative_count
            if negative_count > 0
            else np.nan
        )


        # ROC AUC and average precision require both classes
        # to be represented in the modality-count group.
        if np.unique(
            group_targets
        ).size == 2:

            roc_auc = float(
                roc_auc_score(
                    group_targets,
                    group_probabilities,
                )
            )

            average_precision = float(
                average_precision_score(
                    group_targets,
                    group_probabilities,
                )
            )

            balanced_accuracy = float(
                balanced_accuracy_score(
                    group_targets,
                    group_predictions,
                )
            )

        else:

            roc_auc = np.nan
            average_precision = np.nan
            balanced_accuracy = np.nan


        modality_count_metric_rows.append(
            {
                "ORIGINAL_MODALITY_COUNT":
                    int(
                        modality_count
                    ),

                "OUTPUT":
                    output_name,

                "PARTICIPANTS":
                    int(
                        len(
                            group_targets
                        )
                    ),

                "sMCI_PARTICIPANTS":
                    int(
                        np.sum(
                            group_targets
                            == 0
                        )
                    ),

                "pMCI_PARTICIPANTS":
                    int(
                        np.sum(
                            group_targets
                            == 1
                        )
                    ),

                "ROC_AUC":
                    roc_auc,

                "AVERAGE_PRECISION":
                    average_precision,

                "BALANCED_ACCURACY":
                    balanced_accuracy,

                "SENSITIVITY":
                    float(
                        sensitivity
                    ),

                "SPECIFICITY":
                    float(
                        specificity
                    ),

                "BRIER_SCORE":
                    float(
                        brier_score_loss(
                            group_targets,
                            group_probabilities,
                        )
                    ),

                "MEAN_UNCERTAINTY":
                    float(
                        np.mean(
                            group_uncertainties
                        )
                    ),

                "MEDIAN_UNCERTAINTY":
                    float(
                        np.median(
                            group_uncertainties
                        )
                    ),

                "CORRECT_PREDICTIONS":
                    int(
                        np.sum(
                            group_predictions
                            == group_targets
                        )
                    ),

                "INCORRECT_PREDICTIONS":
                    int(
                        np.sum(
                            group_predictions
                            != group_targets
                        )
                    ),
            }
        )


performance_by_modality_count = pd.DataFrame(
    modality_count_metric_rows
)


performance_by_modality_count[
    "OUTPUT"
] = pd.Categorical(
    performance_by_modality_count[
        "OUTPUT"
    ],
    categories=OUTPUT_ORDER,
    ordered=True,
)


performance_by_modality_count = (
    performance_by_modality_count
    .sort_values(
        [
            "ORIGINAL_MODALITY_COUNT",
            "OUTPUT",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Test whether uncertainty changes monotonically with modality count
# ------------------------------------------------------------

modality_count_uncertainty_rows = []


for output_name in OUTPUT_ORDER:

    uncertainty_column = (
        POOLED_OUTPUT_COLUMNS[
            output_name
        ][
            "uncertainty"
        ]
    )


    correlation_result = spearmanr(
        pooled_out_of_fold_predictions[
            "ORIGINAL_MODALITY_COUNT"
        ],
        pooled_out_of_fold_predictions[
            uncertainty_column
        ],
    )


    modality_count_uncertainty_rows.append(
        {
            "OUTPUT":
                output_name,

            "SPEARMAN_CORRELATION":
                float(
                    correlation_result.statistic
                ),

            "P_VALUE":
                float(
                    correlation_result.pvalue
                ),
        }
    )


modality_count_uncertainty_association = pd.DataFrame(
    modality_count_uncertainty_rows
)


# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

MODALITY_COUNT_DISTRIBUTION_PATH = (
    TABLE_DIR
    / "modality_count_distribution.csv"
)

PERFORMANCE_BY_MODALITY_COUNT_PATH = (
    TABLE_DIR
    / "performance_by_modality_count.csv"
)

MODALITY_COUNT_UNCERTAINTY_PATH = (
    STATISTICS_DIR
    / "modality_count_uncertainty_association.csv"
)


modality_count_distribution.to_csv(
    MODALITY_COUNT_DISTRIBUTION_PATH,
    index=False,
)

performance_by_modality_count.to_csv(
    PERFORMANCE_BY_MODALITY_COUNT_PATH,
    index=False,
)

modality_count_uncertainty_association.to_csv(
    MODALITY_COUNT_UNCERTAINTY_PATH,
    index=False,
)


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("=" * 72)
print("PERFORMANCE AND UNCERTAINTY BY MODALITY COUNT")
print("=" * 72)


print(
    "\nParticipant distribution by available modality count:"
)

display(
    modality_count_distribution.round(
        6
    )
)


print(
    "\nPerformance by modality count:"
)

display(
    performance_by_modality_count[
        [
            "ORIGINAL_MODALITY_COUNT",
            "OUTPUT",
            "PARTICIPANTS",
            "sMCI_PARTICIPANTS",
            "pMCI_PARTICIPANTS",
            "ROC_AUC",
            "AVERAGE_PRECISION",
            "BALANCED_ACCURACY",
            "SENSITIVITY",
            "SPECIFICITY",
            "BRIER_SCORE",
            "MEAN_UNCERTAINTY",
        ]
    ]
    .round(
        6
    )
)


print(
    "\nAssociation between modality count and uncertainty:"
)

display(
    modality_count_uncertainty_association.round(
        6
    )
)


print(
    "\nModality-count distribution saved to:\n"
    f"{MODALITY_COUNT_DISTRIBUTION_PATH}"
)

print(
    "\nPerformance by modality count saved to:\n"
    f"{PERFORMANCE_BY_MODALITY_COUNT_PATH}"
)

print(
    "\nModality-count and uncertainty association saved to:\n"
    f"{MODALITY_COUNT_UNCERTAINTY_PATH}"
)

### 1.8.1. Performance and uncertainty by modality count

The analysis above groups participants by the number of available modality branches and reports performance and uncertainty within each group.

These are descriptive subgroup results. Differences may reflect both information availability and differences in participant composition, so they should not be treated as causal modality-importance estimates.


## 1.9. Evaluating selective prediction using uncertainty

A clinically useful uncertainty estimate should help identify cases for which the model should avoid making an automatic decision.

I therefore evaluate selective prediction by progressively excluding the most uncertain participants.

For each model output, participants are ranked from lowest to highest uncertainty. I then retain only the least uncertain:

- 100% of participants;
- 90%;
- 80%;
- 70%;
- 60%;
- 50%.

At each retained coverage level, I calculate:

- number of retained participants;
- ROC AUC;
- average precision;
- accuracy;
- balanced accuracy;
- sensitivity;
- specificity;
- Brier score;
- error rate;
- mean uncertainty.

Coverage is defined as the proportion of participants for whom the model retains a prediction.

The corresponding rejection rate is:

$$
\text{Rejection rate} = 1 - \text{coverage}
$$

A useful uncertainty estimate should produce lower error rates and generally stronger predictive performance as the most uncertain cases are rejected.

This analysis does not suggest that uncertain participants should be ignored clinically. Instead, it evaluates whether uncertainty can identify cases that may require additional measurements, specialist review, or cautious interpretation.

In [ ]:
# ============================================================
# 9. Evaluating selective prediction using uncertainty
# ============================================================

# ------------------------------------------------------------
# Coverage levels retained after uncertainty-based rejection
# ------------------------------------------------------------

COVERAGE_LEVELS = [
    1.00,
    0.90,
    0.80,
    0.70,
    0.60,
    0.50,
]


# ------------------------------------------------------------
# Calculate selective-prediction metrics
# ------------------------------------------------------------

selective_prediction_rows = []


for output_name in OUTPUT_ORDER:

    probability_column = (
        POOLED_OUTPUT_COLUMNS[
            output_name
        ][
            "probability"
        ]
    )

    uncertainty_column = (
        POOLED_OUTPUT_COLUMNS[
            output_name
        ][
            "uncertainty"
        ]
    )


    output_table = (
        pooled_out_of_fold_predictions[
            [
                "RID",
                "TARGET",
                probability_column,
                uncertainty_column,
            ]
        ]
        .copy()
        .sort_values(
            uncertainty_column,
            ascending=True,
        )
        .reset_index(
            drop=True
        )
    )


    total_participants = len(
        output_table
    )


    for coverage in COVERAGE_LEVELS:

        retained_count = int(
            np.floor(
                total_participants
                * coverage
            )
        )


        retained_count = max(
            retained_count,
            1,
        )


        retained_table = (
            output_table
            .iloc[
                :retained_count
            ]
            .copy()
        )


        retained_targets = (
            retained_table[
                "TARGET"
            ]
            .to_numpy(
                dtype=int
            )
        )

        retained_probabilities = (
            retained_table[
                probability_column
            ]
            .to_numpy(
                dtype=float
            )
        )

        retained_uncertainties = (
            retained_table[
                uncertainty_column
            ]
            .to_numpy(
                dtype=float
            )
        )

        retained_predictions = (
            retained_probabilities
            >= CLASSIFICATION_THRESHOLD
        ).astype(
            int
        )


        retained_confusion = confusion_matrix(
            retained_targets,
            retained_predictions,
            labels=[
                0,
                1,
            ],
        )


        true_negative = int(
            retained_confusion[
                0,
                0,
            ]
        )

        false_positive = int(
            retained_confusion[
                0,
                1,
            ]
        )

        false_negative = int(
            retained_confusion[
                1,
                0,
            ]
        )

        true_positive = int(
            retained_confusion[
                1,
                1,
            ]
        )


        negative_count = (
            true_negative
            + false_positive
        )

        positive_count = (
            true_positive
            + false_negative
        )


        sensitivity = (
            true_positive
            / positive_count
            if positive_count > 0
            else np.nan
        )

        specificity = (
            true_negative
            / negative_count
            if negative_count > 0
            else np.nan
        )


        if np.unique(
            retained_targets
        ).size == 2:

            roc_auc = float(
                roc_auc_score(
                    retained_targets,
                    retained_probabilities,
                )
            )

            average_precision = float(
                average_precision_score(
                    retained_targets,
                    retained_probabilities,
                )
            )

            balanced_accuracy = float(
                balanced_accuracy_score(
                    retained_targets,
                    retained_predictions,
                )
            )

        else:

            roc_auc = np.nan
            average_precision = np.nan
            balanced_accuracy = np.nan


        accuracy = float(
            accuracy_score(
                retained_targets,
                retained_predictions,
            )
        )


        selective_prediction_rows.append(
            {
                "OUTPUT":
                    output_name,

                "COVERAGE":
                    float(
                        retained_count
                        / total_participants
                    ),

                "TARGET_COVERAGE":
                    float(
                        coverage
                    ),

                "REJECTION_RATE":
                    float(
                        1.0
                        -
                        (
                            retained_count
                            / total_participants
                        )
                    ),

                "RETAINED_PARTICIPANTS":
                    int(
                        retained_count
                    ),

                "REJECTED_PARTICIPANTS":
                    int(
                        total_participants
                        - retained_count
                    ),

                "sMCI_PARTICIPANTS":
                    int(
                        np.sum(
                            retained_targets
                            == 0
                        )
                    ),

                "pMCI_PARTICIPANTS":
                    int(
                        np.sum(
                            retained_targets
                            == 1
                        )
                    ),

                "ROC_AUC":
                    roc_auc,

                "AVERAGE_PRECISION":
                    average_precision,

                "ACCURACY":
                    accuracy,

                "BALANCED_ACCURACY":
                    balanced_accuracy,

                "SENSITIVITY":
                    float(
                        sensitivity
                    ),

                "SPECIFICITY":
                    float(
                        specificity
                    ),

                "BRIER_SCORE":
                    float(
                        brier_score_loss(
                            retained_targets,
                            retained_probabilities,
                        )
                    ),

                "ERROR_RATE":
                    float(
                        1.0
                        - accuracy
                    ),

                "MEAN_UNCERTAINTY":
                    float(
                        np.mean(
                            retained_uncertainties
                        )
                    ),

                "MAXIMUM_RETAINED_UNCERTAINTY":
                    float(
                        np.max(
                            retained_uncertainties
                        )
                    ),
            }
        )


selective_prediction_summary = pd.DataFrame(
    selective_prediction_rows
)


# ------------------------------------------------------------
# Preserve the intended output order
# ------------------------------------------------------------

selective_prediction_summary[
    "OUTPUT"
] = pd.Categorical(
    selective_prediction_summary[
        "OUTPUT"
    ],
    categories=OUTPUT_ORDER,
    ordered=True,
)


selective_prediction_summary = (
    selective_prediction_summary
    .sort_values(
        [
            "OUTPUT",
            "COVERAGE",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Calculate improvement relative to full coverage
# ------------------------------------------------------------

full_coverage_reference = (
    selective_prediction_summary.loc[
        selective_prediction_summary[
            "TARGET_COVERAGE"
        ]
        == 1.00,
        [
            "OUTPUT",
            "ERROR_RATE",
            "BRIER_SCORE",
            "BALANCED_ACCURACY",
            "ROC_AUC",
        ],
    ]
    .rename(
        columns={
            "ERROR_RATE":
                "FULL_COVERAGE_ERROR_RATE",

            "BRIER_SCORE":
                "FULL_COVERAGE_BRIER_SCORE",

            "BALANCED_ACCURACY":
                "FULL_COVERAGE_BALANCED_ACCURACY",

            "ROC_AUC":
                "FULL_COVERAGE_ROC_AUC",
        }
    )
)


selective_prediction_summary = (
    selective_prediction_summary
    .merge(
        full_coverage_reference,
        on="OUTPUT",
        how="left",
        validate="many_to_one",
    )
)


selective_prediction_summary[
    "ERROR_RATE_REDUCTION"
] = (
    selective_prediction_summary[
        "FULL_COVERAGE_ERROR_RATE"
    ]
    -
    selective_prediction_summary[
        "ERROR_RATE"
    ]
)


selective_prediction_summary[
    "BRIER_SCORE_REDUCTION"
] = (
    selective_prediction_summary[
        "FULL_COVERAGE_BRIER_SCORE"
    ]
    -
    selective_prediction_summary[
        "BRIER_SCORE"
    ]
)


selective_prediction_summary[
    "BALANCED_ACCURACY_CHANGE"
] = (
    selective_prediction_summary[
        "BALANCED_ACCURACY"
    ]
    -
    selective_prediction_summary[
        "FULL_COVERAGE_BALANCED_ACCURACY"
    ]
)


selective_prediction_summary[
    "ROC_AUC_CHANGE"
] = (
    selective_prediction_summary[
        "ROC_AUC"
    ]
    -
    selective_prediction_summary[
        "FULL_COVERAGE_ROC_AUC"
    ]
)


# ------------------------------------------------------------
# Save selective-prediction results
# ------------------------------------------------------------

SELECTIVE_PREDICTION_PATH = (
    TABLE_DIR
    / "selective_prediction_by_uncertainty.csv"
)


selective_prediction_summary.to_csv(
    SELECTIVE_PREDICTION_PATH,
    index=False,
)


# ------------------------------------------------------------
# Display principal results
# ------------------------------------------------------------

print("=" * 72)
print("SELECTIVE PREDICTION USING UNCERTAINTY")
print("=" * 72)


print(
    "\nPerformance after rejecting the most uncertain cases:"
)

display(
    selective_prediction_summary[
        [
            "OUTPUT",
            "COVERAGE",
            "RETAINED_PARTICIPANTS",
            "REJECTED_PARTICIPANTS",
            "ROC_AUC",
            "BALANCED_ACCURACY",
            "SENSITIVITY",
            "SPECIFICITY",
            "BRIER_SCORE",
            "ERROR_RATE",
            "MEAN_UNCERTAINTY",
        ]
    ]
    .round(
        6
    )
)


print(
    "\nChange relative to retaining all participants:"
)

display(
    selective_prediction_summary[
        [
            "OUTPUT",
            "COVERAGE",
            "ERROR_RATE_REDUCTION",
            "BRIER_SCORE_REDUCTION",
            "BALANCED_ACCURACY_CHANGE",
            "ROC_AUC_CHANGE",
        ]
    ]
    .round(
        6
    )
)


print(
    "\nSelective-prediction results saved to:\n"
    f"{SELECTIVE_PREDICTION_PATH}"
)

### 1.9.1. Selective prediction using uncertainty

The risk-coverage results above test whether removing the most uncertain predictions improves accuracy among the retained participants.

A useful uncertainty signal should reduce prediction risk as coverage decreases, although the strength and smoothness of this relationship must be judged from the gated experiment itself.


## 1.10. Estimating bootstrap confidence intervals and paired performance differences

The previous analyses provide point estimates, but they do not show how precisely performance has been estimated from the 544 participants.

I therefore use participant-level bootstrap resampling to estimate 95% confidence intervals for the pooled out-of-fold metrics.

In each bootstrap iteration:

1. I sample 544 participants with replacement from the pooled out-of-fold prediction table;
2. The notebook preserves the paired predictions from the Hybrid, interaction pathway-only, and evidence pathway-only outputs for every sampled participant;
3. The notebook calculates ROC AUC, average precision, balanced accuracy, and Brier score for each output;
4. The notebook calculates paired metric differences between the outputs.

The paired comparisons are:

- Hybrid minus interaction pathway-only;
- Hybrid minus evidence pathway-only;
- evidence pathway-only minus interaction pathway-only.

Because the same resampled participants are used for all outputs, these comparisons preserve the participant-level pairing between model predictions.

For ROC AUC, average precision, and balanced accuracy, a positive difference favours the first output named in the comparison.

For Brier score, a negative difference favours the first output because a lower Brier score indicates better probabilistic accuracy.

The 95% bootstrap confidence interval is defined using the 2.5th and 97.5th percentiles of the bootstrap distribution.

These intervals quantify sampling uncertainty within the present cohort. They do not represent external validation and should not be interpreted as evidence of performance in an independent clinical population.

In [ ]:
# ============================================================
# 10. Bootstrap confidence intervals and paired comparisons
# ============================================================

# ------------------------------------------------------------
# Bootstrap configuration
# ------------------------------------------------------------

BOOTSTRAP_ITERATIONS = 2000
BOOTSTRAP_RANDOM_SEED = 42

bootstrap_random_generator = np.random.default_rng(
    BOOTSTRAP_RANDOM_SEED
)


# ------------------------------------------------------------
# Metrics included in the bootstrap analysis
# ------------------------------------------------------------

BOOTSTRAP_METRICS = [
    "ROC_AUC",
    "AVERAGE_PRECISION",
    "BALANCED_ACCURACY",
    "BRIER_SCORE",
]


# ------------------------------------------------------------
# Pairwise model comparisons
# ------------------------------------------------------------

PAIRWISE_COMPARISONS = [
    (
        "Hybrid",
        "3MT-only",
    ),
    (
        "Hybrid",
        "TMC-only",
    ),
    (
        "TMC-only",
        "3MT-only",
    ),
]


# ------------------------------------------------------------
# Prepare authoritative arrays once
# ------------------------------------------------------------

bootstrap_targets = (
    pooled_out_of_fold_predictions[
        "TARGET"
    ]
    .to_numpy(
        dtype=int
    )
)


bootstrap_probabilities = {
    output_name:
        pooled_out_of_fold_predictions[
            POOLED_OUTPUT_COLUMNS[
                output_name
            ][
                "probability"
            ]
        ]
        .to_numpy(
            dtype=float
        )

    for output_name in OUTPUT_ORDER
}


participant_count = len(
    bootstrap_targets
)


# ------------------------------------------------------------
# Calculate the four required metrics
# ------------------------------------------------------------

def calculate_bootstrap_metrics(
    targets,
    probabilities,
    threshold=0.50,
):
    """
    Calculate the principal metrics used in the participant-level
    bootstrap analysis.
    """

    predicted_classes = (
        probabilities
        >= threshold
    ).astype(
        int
    )


    return {
        "ROC_AUC":
            float(
                roc_auc_score(
                    targets,
                    probabilities,
                )
            ),

        "AVERAGE_PRECISION":
            float(
                average_precision_score(
                    targets,
                    probabilities,
                )
            ),

        "BALANCED_ACCURACY":
            float(
                balanced_accuracy_score(
                    targets,
                    predicted_classes,
                )
            ),

        "BRIER_SCORE":
            float(
                brier_score_loss(
                    targets,
                    probabilities,
                )
            ),
    }


# ------------------------------------------------------------
# Original pooled point estimates
# ------------------------------------------------------------

original_metric_values = {}


for output_name in OUTPUT_ORDER:

    original_metric_values[
        output_name
    ] = calculate_bootstrap_metrics(
        targets=
            bootstrap_targets,

        probabilities=
            bootstrap_probabilities[
                output_name
            ],

        threshold=
            CLASSIFICATION_THRESHOLD,
    )


# ------------------------------------------------------------
# Containers for bootstrap distributions
# ------------------------------------------------------------

bootstrap_metric_distributions = {
    output_name: {
        metric_name: []
        for metric_name in BOOTSTRAP_METRICS
    }
    for output_name in OUTPUT_ORDER
}


valid_bootstrap_iterations = 0
discarded_single_class_iterations = 0


# ------------------------------------------------------------
# Participant-level paired bootstrap
# ------------------------------------------------------------

for bootstrap_iteration in range(
    BOOTSTRAP_ITERATIONS
):

    sampled_indices = (
        bootstrap_random_generator.integers(
            low=0,
            high=participant_count,
            size=participant_count,
        )
    )


    sampled_targets = bootstrap_targets[
        sampled_indices
    ]


    # ROC AUC requires both classes. A single-class resample is
    # discarded rather than assigned an artificial value.
    if np.unique(
        sampled_targets
    ).size < 2:

        discarded_single_class_iterations += 1
        continue


    for output_name in OUTPUT_ORDER:

        sampled_probabilities = (
            bootstrap_probabilities[
                output_name
            ][
                sampled_indices
            ]
        )


        sampled_metrics = calculate_bootstrap_metrics(
            targets=
                sampled_targets,

            probabilities=
                sampled_probabilities,

            threshold=
                CLASSIFICATION_THRESHOLD,
        )


        for metric_name in BOOTSTRAP_METRICS:

            bootstrap_metric_distributions[
                output_name
            ][
                metric_name
            ].append(
                sampled_metrics[
                    metric_name
                ]
            )


    valid_bootstrap_iterations += 1


# ------------------------------------------------------------
# Summarise confidence intervals for each output
# ------------------------------------------------------------

bootstrap_confidence_interval_rows = []


for output_name in OUTPUT_ORDER:

    for metric_name in BOOTSTRAP_METRICS:

        bootstrap_values = np.asarray(
            bootstrap_metric_distributions[
                output_name
            ][
                metric_name
            ],
            dtype=float,
        )


        bootstrap_confidence_interval_rows.append(
            {
                "OUTPUT":
                    output_name,

                "METRIC":
                    metric_name,

                "POINT_ESTIMATE":
                    float(
                        original_metric_values[
                            output_name
                        ][
                            metric_name
                        ]
                    ),

                "BOOTSTRAP_MEAN":
                    float(
                        np.mean(
                            bootstrap_values
                        )
                    ),

                "BOOTSTRAP_STANDARD_ERROR":
                    float(
                        np.std(
                            bootstrap_values,
                            ddof=1,
                        )
                    ),

                "CI_95_LOWER":
                    float(
                        np.percentile(
                            bootstrap_values,
                            2.5,
                        )
                    ),

                "CI_95_UPPER":
                    float(
                        np.percentile(
                            bootstrap_values,
                            97.5,
                        )
                    ),

                "VALID_BOOTSTRAP_ITERATIONS":
                    int(
                        len(
                            bootstrap_values
                        )
                    ),
            }
        )


bootstrap_confidence_intervals = pd.DataFrame(
    bootstrap_confidence_interval_rows
)


# ------------------------------------------------------------
# Calculate paired bootstrap differences
# ------------------------------------------------------------

paired_bootstrap_rows = []


for first_output, second_output in PAIRWISE_COMPARISONS:

    comparison_name = (
        f"{first_output} minus {second_output}"
    )


    for metric_name in BOOTSTRAP_METRICS:

        first_values = np.asarray(
            bootstrap_metric_distributions[
                first_output
            ][
                metric_name
            ],
            dtype=float,
        )

        second_values = np.asarray(
            bootstrap_metric_distributions[
                second_output
            ][
                metric_name
            ],
            dtype=float,
        )


        paired_differences = (
            first_values
            -
            second_values
        )


        point_difference = (
            original_metric_values[
                first_output
            ][
                metric_name
            ]
            -
            original_metric_values[
                second_output
            ][
                metric_name
            ]
        )


        lower_bound = float(
            np.percentile(
                paired_differences,
                2.5,
            )
        )

        upper_bound = float(
            np.percentile(
                paired_differences,
                97.5,
            )
        )


        # I report the proportion of bootstrap differences on
        # either side of zero as a descriptive two-sided
        # bootstrap probability.
        proportion_at_or_below_zero = float(
            np.mean(
                paired_differences
                <= 0.0
            )
        )

        proportion_at_or_above_zero = float(
            np.mean(
                paired_differences
                >= 0.0
            )
        )

        two_sided_bootstrap_p = min(
            1.0,
            2.0
            * min(
                proportion_at_or_below_zero,
                proportion_at_or_above_zero,
            ),
        )


        paired_bootstrap_rows.append(
            {
                "COMPARISON":
                    comparison_name,

                "FIRST_OUTPUT":
                    first_output,

                "SECOND_OUTPUT":
                    second_output,

                "METRIC":
                    metric_name,

                "POINT_DIFFERENCE":
                    float(
                        point_difference
                    ),

                "BOOTSTRAP_MEAN_DIFFERENCE":
                    float(
                        np.mean(
                            paired_differences
                        )
                    ),

                "BOOTSTRAP_STANDARD_ERROR":
                    float(
                        np.std(
                            paired_differences,
                            ddof=1,
                        )
                    ),

                "CI_95_LOWER":
                    lower_bound,

                "CI_95_UPPER":
                    upper_bound,

                "CI_EXCLUDES_ZERO":
                    bool(
                        (
                            lower_bound
                            > 0.0
                        )
                        or
                        (
                            upper_bound
                            < 0.0
                        )
                    ),

                "DESCRIPTIVE_TWO_SIDED_BOOTSTRAP_P":
                    float(
                        two_sided_bootstrap_p
                    ),

                "VALID_BOOTSTRAP_ITERATIONS":
                    int(
                        len(
                            paired_differences
                        )
                    ),
            }
        )


paired_bootstrap_comparisons = pd.DataFrame(
    paired_bootstrap_rows
)


# ------------------------------------------------------------
# Preserve clear output ordering
# ------------------------------------------------------------

bootstrap_confidence_intervals[
    "OUTPUT"
] = pd.Categorical(
    bootstrap_confidence_intervals[
        "OUTPUT"
    ],
    categories=OUTPUT_ORDER,
    ordered=True,
)


bootstrap_confidence_intervals[
    "METRIC"
] = pd.Categorical(
    bootstrap_confidence_intervals[
        "METRIC"
    ],
    categories=BOOTSTRAP_METRICS,
    ordered=True,
)


bootstrap_confidence_intervals = (
    bootstrap_confidence_intervals
    .sort_values(
        [
            "OUTPUT",
            "METRIC",
        ]
    )
    .reset_index(
        drop=True
    )
)


paired_bootstrap_comparisons[
    "METRIC"
] = pd.Categorical(
    paired_bootstrap_comparisons[
        "METRIC"
    ],
    categories=BOOTSTRAP_METRICS,
    ordered=True,
)


paired_bootstrap_comparisons = (
    paired_bootstrap_comparisons
    .sort_values(
        [
            "COMPARISON",
            "METRIC",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Add thesis-friendly interval strings
# ------------------------------------------------------------

bootstrap_confidence_intervals[
    "POINT_ESTIMATE_WITH_95_CI"
] = (
    bootstrap_confidence_intervals[
        "POINT_ESTIMATE"
    ]
    .map(
        lambda value:
            f"{value:.4f}"
    )
    + " ["
    + bootstrap_confidence_intervals[
        "CI_95_LOWER"
    ]
    .map(
        lambda value:
            f"{value:.4f}"
    )
    + ", "
    + bootstrap_confidence_intervals[
        "CI_95_UPPER"
    ]
    .map(
        lambda value:
            f"{value:.4f}"
    )
    + "]"
)


paired_bootstrap_comparisons[
    "DIFFERENCE_WITH_95_CI"
] = (
    paired_bootstrap_comparisons[
        "POINT_DIFFERENCE"
    ]
    .map(
        lambda value:
            f"{value:+.4f}"
    )
    + " ["
    + paired_bootstrap_comparisons[
        "CI_95_LOWER"
    ]
    .map(
        lambda value:
            f"{value:+.4f}"
    )
    + ", "
    + paired_bootstrap_comparisons[
        "CI_95_UPPER"
    ]
    .map(
        lambda value:
            f"{value:+.4f}"
    )
    + "]"
)


# ------------------------------------------------------------
# Save bootstrap outputs
# ------------------------------------------------------------

BOOTSTRAP_CONFIDENCE_INTERVAL_PATH = (
    STATISTICS_DIR
    / "pooled_metric_bootstrap_confidence_intervals.csv"
)

PAIRED_BOOTSTRAP_COMPARISON_PATH = (
    STATISTICS_DIR
    / "paired_model_bootstrap_comparisons.csv"
)


bootstrap_confidence_intervals.to_csv(
    BOOTSTRAP_CONFIDENCE_INTERVAL_PATH,
    index=False,
)

paired_bootstrap_comparisons.to_csv(
    PAIRED_BOOTSTRAP_COMPARISON_PATH,
    index=False,
)


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("=" * 72)
print("BOOTSTRAP CONFIDENCE INTERVALS AND PAIRED COMPARISONS")
print("=" * 72)


print(
    "\nRequested bootstrap iterations: "
    f"{BOOTSTRAP_ITERATIONS}"
)

print(
    "Valid bootstrap iterations: "
    f"{valid_bootstrap_iterations}"
)

print(
    "Discarded single-class iterations: "
    f"{discarded_single_class_iterations}"
)


print(
    "\nPooled metric estimates with 95% bootstrap confidence intervals:"
)

display(
    bootstrap_confidence_intervals[
        [
            "OUTPUT",
            "METRIC",
            "POINT_ESTIMATE",
            "CI_95_LOWER",
            "CI_95_UPPER",
            "POINT_ESTIMATE_WITH_95_CI",
        ]
    ]
    .round(
        6
    )
)


print(
    "\nPaired model differences with 95% bootstrap confidence intervals:"
)

display(
    paired_bootstrap_comparisons[
        [
            "COMPARISON",
            "METRIC",
            "POINT_DIFFERENCE",
            "CI_95_LOWER",
            "CI_95_UPPER",
            "CI_EXCLUDES_ZERO",
            "DESCRIPTIVE_TWO_SIDED_BOOTSTRAP_P",
        ]
    ]
    .round(
        6
    )
)


print(
    "\nBootstrap confidence intervals saved to:\n"
    f"{BOOTSTRAP_CONFIDENCE_INTERVAL_PATH}"
)

print(
    "\nPaired bootstrap comparisons saved to:\n"
    f"{PAIRED_BOOTSTRAP_COMPARISON_PATH}"
)

### 1.10.1. Bootstrap confidence intervals and paired output comparisons

The bootstrap results above quantify uncertainty around the pooled metrics and paired differences among the Hybrid, interaction pathway-only, and evidence pathway-only outputs within this pre-baseline temporal-sensitivity experiment.

A separate cross-experiment comparison is still required to test fixed equal fusion directly against the learned-gate experiment using matched participant-level predictions.


## 1.11. Detailed calibration analysis

The pooled evaluation already included Brier score and expected calibration error. examine calibration in more detail.

Calibration describes whether predicted probabilities correspond to observed outcome frequencies. For example, among participants assigned a pMCI probability close to \(0.70\), approximately 70% should belong to the pMCI class in a well-calibrated model.

For each output, I divide the predicted probabilities into ten equal-width intervals between 0 and 1. Within each interval, I calculate:

- the number of participants;
- the mean predicted pMCI probability;
- the observed pMCI proportion;
- the signed calibration gap;
- the absolute calibration gap.

The signed calibration gap is defined as:

$$
\text{Calibration gap}
=
\text{Observed pMCI proportion}
-
\text{Mean predicted pMCI probability}
$$

A positive gap indicates that the model underestimated pMCI risk within that interval. A negative gap indicates that it overestimated pMCI risk.

I also estimate a calibration intercept and calibration slope from the pooled predictions.

For an ideally calibrated model:

$$
\text{Calibration intercept} = 0
$$

and

$$
\text{Calibration slope} = 1
$$

A positive intercept suggests general underprediction of pMCI risk, whereas a negative intercept suggests general overprediction.

A calibration slope below 1 commonly indicates that predictions are too extreme. A slope above 1 indicates that predictions are not sufficiently separated or are too close to the overall outcome prevalence.

In [ ]:
# ============================================================
# 11. Detailed pooled calibration analysis
# ============================================================

from scipy.optimize import minimize


# ------------------------------------------------------------
# Calibration configuration
# ------------------------------------------------------------

DETAILED_CALIBRATION_BIN_COUNT = 10

calibration_bin_edges = np.linspace(
    0.0,
    1.0,
    DETAILED_CALIBRATION_BIN_COUNT + 1,
)


# ------------------------------------------------------------
# Estimate calibration intercept and slope
# ------------------------------------------------------------

def estimate_calibration_intercept_and_slope(
    targets,
    probabilities,
):
    """
    Estimate the calibration intercept and slope by fitting:

        logit(P(Y = 1)) = intercept + slope * logit(predicted probability)

    I fit the two parameters by minimising binary negative
    log-likelihood.
    """

    clipped_probabilities = np.clip(
        probabilities,
        1e-7,
        1.0 - 1e-7,
    )


    predicted_logits = np.log(
        clipped_probabilities
        /
        (
            1.0
            - clipped_probabilities
        )
    )


    def calibration_negative_log_likelihood(
        parameters,
    ):

        intercept = parameters[
            0
        ]

        slope = parameters[
            1
        ]


        calibrated_logits = (
            intercept
            +
            slope
            * predicted_logits
        )


        # I calculate the logistic function in a numerically
        # stable form.
        calibrated_probabilities = np.where(
            calibrated_logits
            >= 0.0,

            1.0
            /
            (
                1.0
                +
                np.exp(
                    -calibrated_logits
                )
            ),

            np.exp(
                calibrated_logits
            )
            /
            (
                1.0
                +
                np.exp(
                    calibrated_logits
                )
            ),
        )


        calibrated_probabilities = np.clip(
            calibrated_probabilities,
            1e-12,
            1.0 - 1e-12,
        )


        negative_log_likelihood = -np.sum(
            targets
            * np.log(
                calibrated_probabilities
            )
            +
            (
                1
                - targets
            )
            * np.log(
                1.0
                - calibrated_probabilities
            )
        )


        return float(
            negative_log_likelihood
        )


    optimisation_result = minimize(
        calibration_negative_log_likelihood,
        x0=np.array(
            [
                0.0,
                1.0,
            ],
            dtype=float,
        ),
        method="BFGS",
    )


    if not optimisation_result.success:

        raise RuntimeError(
            "Calibration intercept and slope optimisation failed: "
            f"{optimisation_result.message}"
        )


    return {
        "CALIBRATION_INTERCEPT":
            float(
                optimisation_result.x[
                    0
                ]
            ),

        "CALIBRATION_SLOPE":
            float(
                optimisation_result.x[
                    1
                ]
            ),
    }


# ------------------------------------------------------------
# Calculate reliability-bin tables
# ------------------------------------------------------------

calibration_bin_rows = []
calibration_summary_rows = []


for output_name in OUTPUT_ORDER:

    probability_column = (
        POOLED_OUTPUT_COLUMNS[
            output_name
        ][
            "probability"
        ]
    )


    probabilities = (
        pooled_out_of_fold_predictions[
            probability_column
        ]
        .to_numpy(
            dtype=float
        )
    )


    # --------------------------------------------------------
    # Assign every participant to one probability interval
    # --------------------------------------------------------

    probability_bin_indices = np.digitize(
        probabilities,
        calibration_bin_edges[
            1:-1
        ],
        right=False,
    )


    for bin_index in range(
        DETAILED_CALIBRATION_BIN_COUNT
    ):

        bin_mask = (
            probability_bin_indices
            == bin_index
        )


        bin_participants = int(
            np.sum(
                bin_mask
            )
        )


        lower_bound = float(
            calibration_bin_edges[
                bin_index
            ]
        )

        upper_bound = float(
            calibration_bin_edges[
                bin_index + 1
            ]
        )


        if bin_participants > 0:

            mean_predicted_probability = float(
                np.mean(
                    probabilities[
                        bin_mask
                    ]
                )
            )

            observed_pMCI_proportion = float(
                np.mean(
                    pooled_targets[
                        bin_mask
                    ]
                )
            )

            calibration_gap = (
                observed_pMCI_proportion
                -
                mean_predicted_probability
            )

            absolute_calibration_gap = abs(
                calibration_gap
            )

        else:

            mean_predicted_probability = np.nan
            observed_pMCI_proportion = np.nan
            calibration_gap = np.nan
            absolute_calibration_gap = np.nan


        calibration_bin_rows.append(
            {
                "OUTPUT":
                    output_name,

                "BIN_INDEX":
                    int(
                        bin_index + 1
                    ),

                "BIN_LOWER_BOUND":
                    lower_bound,

                "BIN_UPPER_BOUND":
                    upper_bound,

                "PARTICIPANTS":
                    bin_participants,

                "MEAN_PREDICTED_pMCI_PROBABILITY":
                    mean_predicted_probability,

                "OBSERVED_pMCI_PROPORTION":
                    observed_pMCI_proportion,

                "CALIBRATION_GAP":
                    calibration_gap,

                "ABSOLUTE_CALIBRATION_GAP":
                    absolute_calibration_gap,
            }
        )


    # --------------------------------------------------------
    # Estimate calibration intercept and slope
    # --------------------------------------------------------

    calibration_parameters = (
        estimate_calibration_intercept_and_slope(
            targets=
                pooled_targets,

            probabilities=
                probabilities,
        )
    )


    # --------------------------------------------------------
    # Recalculate summary measures from pooled predictions
    # --------------------------------------------------------

    output_ece = (
        calculate_expected_calibration_error(
            targets=
                pooled_targets,

            probabilities=
                probabilities,

            number_of_bins=
                DETAILED_CALIBRATION_BIN_COUNT,
        )
    )


    output_brier_score = float(
        brier_score_loss(
            pooled_targets,
            probabilities,
        )
    )


    calibration_summary_rows.append(
        {
            "OUTPUT":
                output_name,

            "PARTICIPANTS":
                int(
                    len(
                        pooled_targets
                    )
                ),

            "OBSERVED_pMCI_PREVALENCE":
                float(
                    np.mean(
                        pooled_targets
                    )
                ),

            "MEAN_PREDICTED_pMCI_PROBABILITY":
                float(
                    np.mean(
                        probabilities
                    )
                ),

            "MEAN_PREDICTION_MINUS_PREVALENCE":
                float(
                    np.mean(
                        probabilities
                    )
                    -
                    np.mean(
                        pooled_targets
                    )
                ),

            "BRIER_SCORE":
                output_brier_score,

            "EXPECTED_CALIBRATION_ERROR":
                output_ece,

            "CALIBRATION_INTERCEPT":
                calibration_parameters[
                    "CALIBRATION_INTERCEPT"
                ],

            "CALIBRATION_SLOPE":
                calibration_parameters[
                    "CALIBRATION_SLOPE"
                ],
        }
    )


# ------------------------------------------------------------
# Final calibration tables
# ------------------------------------------------------------

calibration_bin_summary = pd.DataFrame(
    calibration_bin_rows
)

detailed_calibration_summary = pd.DataFrame(
    calibration_summary_rows
)


calibration_bin_summary[
    "OUTPUT"
] = pd.Categorical(
    calibration_bin_summary[
        "OUTPUT"
    ],
    categories=OUTPUT_ORDER,
    ordered=True,
)


detailed_calibration_summary[
    "OUTPUT"
] = pd.Categorical(
    detailed_calibration_summary[
        "OUTPUT"
    ],
    categories=OUTPUT_ORDER,
    ordered=True,
)


calibration_bin_summary = (
    calibration_bin_summary
    .sort_values(
        [
            "OUTPUT",
            "BIN_INDEX",
        ]
    )
    .reset_index(
        drop=True
    )
)


detailed_calibration_summary = (
    detailed_calibration_summary
    .sort_values(
        "OUTPUT"
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Check that each model's bins contain all 544 participants
# ------------------------------------------------------------

calibration_bin_count_check = (
    calibration_bin_summary
    .groupby(
        "OUTPUT",
        observed=False,
    )[
        "PARTICIPANTS"
    ]
    .sum()
    .reset_index(
        name="BINNED_PARTICIPANTS"
    )
)


calibration_bin_count_check[
    "EXPECTED_PARTICIPANTS"
] = len(
    pooled_targets
)


calibration_bin_count_check[
    "COUNT_MATCH"
] = (
    calibration_bin_count_check[
        "BINNED_PARTICIPANTS"
    ]
    ==
    calibration_bin_count_check[
        "EXPECTED_PARTICIPANTS"
    ]
)


if not calibration_bin_count_check[
    "COUNT_MATCH"
].all():

    raise ValueError(
        "At least one model output does not have all participants "
        "represented in the calibration bins."
    )


# ------------------------------------------------------------
# Save calibration tables
# ------------------------------------------------------------

CALIBRATION_BIN_SUMMARY_PATH = (
    TABLE_DIR
    / "pooled_calibration_bins.csv"
)

DETAILED_CALIBRATION_SUMMARY_PATH = (
    TABLE_DIR
    / "pooled_detailed_calibration_summary.csv"
)


calibration_bin_summary.to_csv(
    CALIBRATION_BIN_SUMMARY_PATH,
    index=False,
)

detailed_calibration_summary.to_csv(
    DETAILED_CALIBRATION_SUMMARY_PATH,
    index=False,
)


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("=" * 72)
print("DETAILED POOLED CALIBRATION ANALYSIS")
print("=" * 72)


print(
    "\nCalibration summary:"
)

display(
    detailed_calibration_summary.round(
        6
    )
)


print(
    "\nCalibration-bin participant-count check:"
)

display(
    calibration_bin_count_check
)


print(
    "\nReliability-bin results:"
)

display(
    calibration_bin_summary[
        [
            "OUTPUT",
            "BIN_INDEX",
            "BIN_LOWER_BOUND",
            "BIN_UPPER_BOUND",
            "PARTICIPANTS",
            "MEAN_PREDICTED_pMCI_PROBABILITY",
            "OBSERVED_pMCI_PROPORTION",
            "CALIBRATION_GAP",
        ]
    ]
    .round(
        6
    )
)


print(
    "\nCalibration-bin results saved to:\n"
    f"{CALIBRATION_BIN_SUMMARY_PATH}"
)

print(
    "\nDetailed calibration summary saved to:\n"
    f"{DETAILED_CALIBRATION_SUMMARY_PATH}"
)

### 1.11.1. Detailed pooled calibration analysis

The calibration tables and figures above describe how closely predicted probabilities align with observed outcomes for the gated experiment.

The reported calibration intercept, slope, Brier score, negative log-likelihood, expected calibration error, and reliability bins are descriptive out-of-fold evaluation results. Any later recalibration procedure must be fitted without using held-out test outcomes.


## 1.12. Locating participant-level modality-availability information

The file `final_task_ready_input_inventory.csv` is not a participant-level table. It contains one inventory row for each prepared task and outer fold, rather than one row for each participant.

Its columns describe:

- the task;
- the outer fold;
- the source and prepared output paths;
- train, validation, and test row counts;
- feature counts;
- branch-mask counts;
- feature-mask counts.

I therefore use this inventory to locate the actual participant-level prepared input files.

In this step, I:

- restrict the inventory to the `mci_prognosis` task;
- verify that all five outer folds are represented;
- inspect the paths recorded for each fold;
- load a small sample from each referenced prepared file;
- identify which files contain `RID`;
- identify candidate branch-level or modality-level mask columns.

No missingness-pattern analysis is performed yet. This cell only identifies the correct participant-level source and validates its schema.

In [ ]:
# ============================================================
# 12. Locating participant-level modality-availability data
# ============================================================

# ------------------------------------------------------------
# The temporal experiment created its own five model-ready
# fold tables. These are the authoritative participant-level
# inputs for missingness-pattern analysis.
# ------------------------------------------------------------

TEMPORAL_INPUT_ROOT = (
    MODEL_ROOT
    / "temporal_leakage_sensitivity"
    / "prebaseline_only"
    / "final_task_ready_inputs"
    / "mci_prognosis"
)


inventory_rows = []


for outer_fold in OUTER_FOLDS:

    source_path = (
        TEMPORAL_INPUT_ROOT
        / (
            "mci_prognosis_outer_fold_"
            f"{outer_fold}_final_task_ready.csv"
        )
    )


    if not source_path.is_file():
        raise FileNotFoundError(
            f"Missing temporal fold input:\n"
            f"{source_path}"
        )


    fold_table = pd.read_csv(
        source_path,
        low_memory=False,
    )


    inventory_rows.append(
        {
            "TASK":
                "mci_prognosis",

            "OUTER_FOLD":
                outer_fold,

            "SOURCE_PATH":
                str(
                    source_path
                ),

            "FINAL_OUTPUT_PATH":
                str(
                    source_path
                ),

            "PARTICIPANT_ROWS":
                len(
                    fold_table
                ),

            "TRAIN_ROWS":
                int(
                    (
                        fold_table[
                            "DATA_ROLE"
                        ]
                        .astype(
                            str
                        )
                        .str.strip()
                        .str.lower()
                        == "train"
                    ).sum()
                ),

            "VALIDATION_ROWS":
                int(
                    (
                        fold_table[
                            "DATA_ROLE"
                        ]
                        .astype(
                            str
                        )
                        .str.strip()
                        .str.lower()
                        == "validation"
                    ).sum()
                ),

            "TEST_ROWS":
                int(
                    (
                        fold_table[
                            "DATA_ROLE"
                        ]
                        .astype(
                            str
                        )
                        .str.strip()
                        .str.lower()
                        == "test"
                    ).sum()
                ),

            "BRANCH_MASK_COUNT":
                len(
                    [
                        column
                        for column in fold_table.columns
                        if column.startswith(
                            "BRANCH_MASK__"
                        )
                    ]
                ),

            "FEATURE_MASK_COUNT":
                len(
                    [
                        column
                        for column in fold_table.columns
                        if column.startswith(
                            "FEATURE_MASK__"
                        )
                    ]
                ),
        }
    )


mci_input_inventory = (
    pd.DataFrame(
        inventory_rows
    )
    .sort_values(
        "OUTER_FOLD"
    )
    .reset_index(
        drop=True
    )
)


print("=" * 72)
print("TEMPORAL PARTICIPANT-LEVEL INPUT INVENTORY")
print("=" * 72)


display(
    mci_input_inventory[
        [
            "TASK",
            "OUTER_FOLD",
            "SOURCE_PATH",
            "PARTICIPANT_ROWS",
            "TRAIN_ROWS",
            "VALIDATION_ROWS",
            "TEST_ROWS",
            "BRANCH_MASK_COUNT",
            "FEATURE_MASK_COUNT",
        ]
    ]
)


observed_outer_folds = set(
    pd.to_numeric(
        mci_input_inventory[
            "OUTER_FOLD"
        ],
        errors="raise",
    ).astype(
        int
    )
)


if observed_outer_folds != set(
    OUTER_FOLDS
):
    raise ValueError(
        "Temporal input inventory does not contain "
        "exactly outer folds 0 to 4."
    )


if not (
    mci_input_inventory[
        "PARTICIPANT_ROWS"
    ]
    == 544
).all():
    raise ValueError(
        "At least one temporal fold input does not "
        "contain the expected 544 participants."
    )


print(
    "\nAll five temporal fold inputs are present "
    "and contain 544 participants."
)

## 1.13. Constructing exact modality-availability patterns

The participant-level prepared inputs contain six branch-level availability indicators:

- demographics;
- cognitive and functional assessments;
- CSF biomarkers;
- plasma biomarkers;
- APOE genotype;
- MRI.

load the test participants from each outer fold and combine them into one out-of-fold modality-availability table.

For every participant, I construct:

- the number of available modality branches;
- a readable label listing the available modalities;
- a readable label listing the missing modalities;
- a six-character binary pattern in a fixed modality order.

The fixed pattern order is:

$$
[\text{Demographics},\ \text{Cognitive/Functional},\ \text{CSF},\
\text{Plasma},\ \text{APOE},\ \text{MRI}]
$$

For example, the pattern `110011` means that Demographics, Cognitive/Functional, APOE, and MRI are available, while CSF and Plasma are missing.

I then merge these patterns with the pooled out-of-fold predictions and verify that:

- all 544 participants are represented exactly once;
- each participant comes from the correct test fold;
- the target agrees between the prepared input and prediction tables;
- the reconstructed number of available modalities agrees with `ORIGINAL_MODALITY_COUNT`.

This cell prepares and validates the exact missingness patterns. Performance comparisons by pattern will be performed in the following step.

In [ ]:
# ------------------------------------------------------------
# Define output directories used by this aggregation notebook
# ------------------------------------------------------------

POOLED_PREDICTIONS_DIR = (
    AGGREGATION_ROOT
    / "pooled_predictions"
)

TABLE_DIR = (
    AGGREGATION_ROOT
    / "tables"
)


POOLED_PREDICTIONS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

In [ ]:
# ============================================================
# 13. Constructing exact modality-availability patterns
# ============================================================

# ------------------------------------------------------------
# Define the six branch-level modality indicators
# ------------------------------------------------------------

BRANCH_MASK_COLUMNS = [
    "BRANCH_MASK__DEMOGRAPHICS",
    "BRANCH_MASK__COGNITIVE_FUNCTIONAL",
    "BRANCH_MASK__CSF",
    "BRANCH_MASK__PLASMA",
    "BRANCH_MASK__APOE",
    "BRANCH_MASK__MRI",
]


BRANCH_DISPLAY_NAMES = {
    "BRANCH_MASK__DEMOGRAPHICS":
        "Demographics",

    "BRANCH_MASK__COGNITIVE_FUNCTIONAL":
        "Cognitive/Functional",

    "BRANCH_MASK__CSF":
        "CSF",

    "BRANCH_MASK__PLASMA":
        "Plasma",

    "BRANCH_MASK__APOE":
        "APOE",

    "BRANCH_MASK__MRI":
        "MRI",
}


# ------------------------------------------------------------
# Load the held-out test partition from each outer fold
# ------------------------------------------------------------

fold_test_availability_tables = []


for _, inventory_row in mci_input_inventory.iterrows():

    outer_fold = int(
        inventory_row[
            "OUTER_FOLD"
        ]
    )


    source_path = Path(
        str(
            inventory_row[
                "SOURCE_PATH"
            ]
        )
    )


    required_columns = [
        "RID",
        "PTID",
        "MCI_PROGNOSIS_TARGET",
        "OUTER_FOLD",
        "DATA_ROLE",
        *BRANCH_MASK_COLUMNS,
    ]


    fold_table = pd.read_csv(
        source_path,
        usecols=required_columns,
    )


    # I retain only the participants used as the held-out test
    # partition for the current outer fold.
    fold_test_table = (
        fold_table.loc[
            (
                pd.to_numeric(
                    fold_table[
                        "OUTER_FOLD"
                    ],
                    errors="raise",
                )
                == outer_fold
            )
            &
            (
                fold_table[
                    "DATA_ROLE"
                ]
                .astype(
                    str
                )
                .str.strip()
                .str.lower()
                == "test"
            )
        ]
        .copy()
    )


    fold_test_table[
        "SOURCE_OUTER_FOLD"
    ] = outer_fold


    fold_test_availability_tables.append(
        fold_test_table
    )


# ------------------------------------------------------------
# Combine all five held-out test partitions
# ------------------------------------------------------------

out_of_fold_availability = pd.concat(
    fold_test_availability_tables,
    ignore_index=True,
)


out_of_fold_availability[
    "RID"
] = pd.to_numeric(
    out_of_fold_availability[
        "RID"
    ],
    errors="raise",
).astype(
    int
)


out_of_fold_availability[
    "SOURCE_OUTER_FOLD"
] = pd.to_numeric(
    out_of_fold_availability[
        "SOURCE_OUTER_FOLD"
    ],
    errors="raise",
).astype(
    int
)


# ------------------------------------------------------------
# Convert and validate the binary branch-mask columns
# ------------------------------------------------------------

for column_name in BRANCH_MASK_COLUMNS:

    out_of_fold_availability[
        column_name
    ] = pd.to_numeric(
        out_of_fold_availability[
            column_name
        ],
        errors="raise",
    ).astype(
        int
    )


    invalid_mask_values = set(
        out_of_fold_availability[
            column_name
        ].unique()
    ) - {
        0,
        1,
    }


    if invalid_mask_values:

        raise ValueError(
            f"{column_name} contains values outside 0 and 1: "
            f"{sorted(invalid_mask_values)}"
        )


# ------------------------------------------------------------
# Convert and validate the already numeric target
# ------------------------------------------------------------

out_of_fold_availability[
    "PREPARED_TARGET"
] = pd.to_numeric(
    out_of_fold_availability[
        "MCI_PROGNOSIS_TARGET"
    ],
    errors="raise",
).astype(
    int
)


invalid_target_values = set(
    out_of_fold_availability[
        "PREPARED_TARGET"
    ].unique()
) - {
    0,
    1,
}


if invalid_target_values:

    raise ValueError(
        "Unexpected prepared target values were found: "
        f"{sorted(invalid_target_values)}"
    )


# ------------------------------------------------------------
# Construct participant-level modality patterns
# ------------------------------------------------------------

out_of_fold_availability[
    "RECONSTRUCTED_MODALITY_COUNT"
] = (
    out_of_fold_availability[
        BRANCH_MASK_COLUMNS
    ]
    .sum(
        axis=1
    )
    .astype(
        int
    )
)


def construct_binary_pattern(
    row,
):
    """
    Construct a six-character binary pattern in this fixed order:

    Demographics, Cognitive/Functional, CSF, Plasma, APOE, MRI.
    """

    return "".join(
        str(
            int(
                row[
                    column_name
                ]
            )
        )
        for column_name in BRANCH_MASK_COLUMNS
    )


def construct_available_modalities(
    row,
):
    """
    Return a readable list of the modality branches available
    for one participant.
    """

    available_modalities = [
        BRANCH_DISPLAY_NAMES[
            column_name
        ]
        for column_name in BRANCH_MASK_COLUMNS
        if int(
            row[
                column_name
            ]
        ) == 1
    ]


    return " + ".join(
        available_modalities
    )


def construct_missing_modalities(
    row,
):
    """
    Return a readable list of the modality branches missing for
    one participant.
    """

    missing_modalities = [
        BRANCH_DISPLAY_NAMES[
            column_name
        ]
        for column_name in BRANCH_MASK_COLUMNS
        if int(
            row[
                column_name
            ]
        ) == 0
    ]


    if len(
        missing_modalities
    ) == 0:

        return "None"


    return " + ".join(
        missing_modalities
    )


out_of_fold_availability[
    "MODALITY_PATTERN_BINARY"
] = out_of_fold_availability.apply(
    construct_binary_pattern,
    axis=1,
)


out_of_fold_availability[
    "AVAILABLE_MODALITIES"
] = out_of_fold_availability.apply(
    construct_available_modalities,
    axis=1,
)


out_of_fold_availability[
    "MISSING_MODALITIES"
] = out_of_fold_availability.apply(
    construct_missing_modalities,
    axis=1,
)


# ------------------------------------------------------------
# Validate the reconstructed out-of-fold participant table
# ------------------------------------------------------------

duplicated_test_rids = (
    out_of_fold_availability[
        "RID"
    ]
    .duplicated(
        keep=False
    )
)


if duplicated_test_rids.any():

    display(
        out_of_fold_availability.loc[
            duplicated_test_rids
        ]
        .sort_values(
            "RID"
        )
    )


    raise ValueError(
        "At least one RID appears in more than one held-out "
        "test partition."
    )


if len(
    out_of_fold_availability
) != len(
    pooled_out_of_fold_predictions
):

    raise ValueError(
        "The reconstructed availability table and pooled "
        "prediction table contain different participant counts."
    )


# ------------------------------------------------------------
# Merge modality patterns with pooled test predictions
# ------------------------------------------------------------

pooled_predictions_with_patterns = (
    pooled_out_of_fold_predictions
    .merge(
        out_of_fold_availability[
            [
                "RID",
                "PTID",
                "SOURCE_OUTER_FOLD",
                "PREPARED_TARGET",
                *BRANCH_MASK_COLUMNS,
                "RECONSTRUCTED_MODALITY_COUNT",
                "MODALITY_PATTERN_BINARY",
                "AVAILABLE_MODALITIES",
                "MISSING_MODALITIES",
            ]
        ],
        on="RID",
        how="left",
        validate="one_to_one",
    )
)


# ------------------------------------------------------------
# Validate the merged participant-level information
# ------------------------------------------------------------

missing_pattern_rows = int(
    pooled_predictions_with_patterns[
        "MODALITY_PATTERN_BINARY"
    ]
    .isna()
    .sum()
)


fold_mismatch_count = int(
    (
        pd.to_numeric(
            pooled_predictions_with_patterns[
                "FOLD"
            ],
            errors="raise",
        )
        !=
        pd.to_numeric(
            pooled_predictions_with_patterns[
                "SOURCE_OUTER_FOLD"
            ],
            errors="raise",
        )
    )
    .sum()
)


target_mismatch_count = int(
    (
        pd.to_numeric(
            pooled_predictions_with_patterns[
                "TARGET"
            ],
            errors="raise",
        )
        !=
        pd.to_numeric(
            pooled_predictions_with_patterns[
                "PREPARED_TARGET"
            ],
            errors="raise",
        )
    )
    .sum()
)


modality_count_mismatch_count = int(
    (
        pd.to_numeric(
            pooled_predictions_with_patterns[
                "ORIGINAL_MODALITY_COUNT"
            ],
            errors="raise",
        )
        !=
        pd.to_numeric(
            pooled_predictions_with_patterns[
                "RECONSTRUCTED_MODALITY_COUNT"
            ],
            errors="raise",
        )
    )
    .sum()
)


if missing_pattern_rows > 0:

    raise ValueError(
        "At least one pooled participant could not be assigned "
        "a modality pattern."
    )


if fold_mismatch_count > 0:

    raise ValueError(
        "At least one prediction fold does not match the "
        "prepared-input test fold."
    )


if target_mismatch_count > 0:

    raise ValueError(
        "At least one prepared target does not match the pooled "
        "prediction target."
    )


if modality_count_mismatch_count > 0:

    raise ValueError(
        "At least one reconstructed modality count does not "
        "match ORIGINAL_MODALITY_COUNT."
    )


# ------------------------------------------------------------
# Summarise the exact modality patterns
# ------------------------------------------------------------

exact_pattern_distribution = (
    pooled_predictions_with_patterns
    .groupby(
        [
            "MODALITY_PATTERN_BINARY",
            "RECONSTRUCTED_MODALITY_COUNT",
            "AVAILABLE_MODALITIES",
            "MISSING_MODALITIES",
        ],
        dropna=False,
    )
    .agg(
        PARTICIPANTS=(
            "RID",
            "size",
        ),

        sMCI_PARTICIPANTS=(
            "TARGET",
            lambda values:
                int(
                    np.sum(
                        values
                        == 0
                    )
                ),
        ),

        pMCI_PARTICIPANTS=(
            "TARGET",
            lambda values:
                int(
                    np.sum(
                        values
                        == 1
                    )
                ),
        ),

        pMCI_PROPORTION=(
            "TARGET",
            "mean",
        ),
    )
    .reset_index()
    .sort_values(
        [
            "PARTICIPANTS",
            "MODALITY_PATTERN_BINARY",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(
        drop=True
    )
)


exact_pattern_distribution[
    "COHORT_PROPORTION"
] = (
    exact_pattern_distribution[
        "PARTICIPANTS"
    ]
    /
    len(
        pooled_predictions_with_patterns
    )
)


# ------------------------------------------------------------
# Save the validated participant and pattern tables
# ------------------------------------------------------------

POOLED_PREDICTIONS_WITH_PATTERNS_PATH = (
    POOLED_PREDICTIONS_DIR
    / "pooled_out_of_fold_predictions_with_modality_patterns.csv"
)


EXACT_PATTERN_DISTRIBUTION_PATH = (
    TABLE_DIR
    / "exact_modality_pattern_distribution.csv"
)


pooled_predictions_with_patterns.to_csv(
    POOLED_PREDICTIONS_WITH_PATTERNS_PATH,
    index=False,
)


exact_pattern_distribution.to_csv(
    EXACT_PATTERN_DISTRIBUTION_PATH,
    index=False,
)


# ------------------------------------------------------------
# Display validation results and pattern frequencies
# ------------------------------------------------------------

print("=" * 72)
print("EXACT MODALITY-AVAILABILITY PATTERN CONSTRUCTION")
print("=" * 72)


validation_summary = pd.DataFrame(
    {
        "CHECK": [
            "Pooled participants",
            "Unique pooled RIDs",
            "Missing modality patterns",
            "Fold mismatches",
            "Target mismatches",
            "Modality-count mismatches",
        ],

        "VALUE": [
            len(
                pooled_predictions_with_patterns
            ),

            pooled_predictions_with_patterns[
                "RID"
            ].nunique(),

            missing_pattern_rows,

            fold_mismatch_count,

            target_mismatch_count,

            modality_count_mismatch_count,
        ],
    }
)


print(
    "\nParticipant-level validation:"
)

display(
    validation_summary
)


print(
    "\nExact modality-pattern distribution:"
)

display(
    exact_pattern_distribution[
        [
            "MODALITY_PATTERN_BINARY",
            "RECONSTRUCTED_MODALITY_COUNT",
            "AVAILABLE_MODALITIES",
            "MISSING_MODALITIES",
            "PARTICIPANTS",
            "sMCI_PARTICIPANTS",
            "pMCI_PARTICIPANTS",
            "pMCI_PROPORTION",
            "COHORT_PROPORTION",
        ]
    ]
    .round(
        6
    )
)


print(
    "\nPredictions with modality patterns saved to:\n"
    f"{POOLED_PREDICTIONS_WITH_PATTERNS_PATH}"
)


print(
    "\nExact pattern distribution saved to:\n"
    f"{EXACT_PATTERN_DISTRIBUTION_PATH}"
)

### 1.13.1. Exact modality-availability pattern analysis

The tables above reconstruct the exact six-branch availability pattern for every held-out participant and join it to the gated out-of-fold predictions.

Pattern-level performance is reported only for sufficiently populated groups containing both classes. Because naturally occurring missingness patterns differ in participant composition and disease prevalence, these results should be interpreted as descriptive robustness evidence rather than causal modality importance.


## 1.14. Analysing performance by exact modality pattern

The previous step identified 13 distinct modality-availability patterns.

However, several patterns contain too few participants for reliable standalone performance estimates. I therefore restrict the main pattern-level comparison to patterns that satisfy both of the following conditions:

- at least 20 participants;
- at least one sMCI and one pMCI participant.

Patterns that do not meet these criteria remain in the descriptive distribution table but are excluded from the main metric comparison.

For each eligible modality pattern, The notebook calculates separately for the Hybrid, interaction pathway-only, and evidence pathway-only outputs:

- participant count;
- sMCI and pMCI counts;
- pMCI prevalence;
- ROC AUC;
- average precision;
- balanced accuracy;
- sensitivity;
- specificity;
- Brier score;
- error rate;
- mean uncertainty.

For the Hybrid output, I also summarise the mean reliability-gate weights assigned to the interaction pathway and modality-specific evidence pathways.

These results describe model behaviour within naturally occurring missingness groups. They should not be interpreted as causal estimates of the effect of removing a modality because the groups differ in sample size, class balance, and potentially other participant characteristics.

In [ ]:
# ============================================================
# 14. Analysing performance by exact modality pattern
# ============================================================

# ------------------------------------------------------------
# Pattern inclusion criteria
# ------------------------------------------------------------

MINIMUM_PATTERN_PARTICIPANTS = 20


# ------------------------------------------------------------
# Identify patterns suitable for metric estimation
# ------------------------------------------------------------

eligible_pattern_table = (
    exact_pattern_distribution.loc[
        (
            exact_pattern_distribution[
                "PARTICIPANTS"
            ]
            >= MINIMUM_PATTERN_PARTICIPANTS
        )
        &
        (
            exact_pattern_distribution[
                "sMCI_PARTICIPANTS"
            ]
            > 0
        )
        &
        (
            exact_pattern_distribution[
                "pMCI_PARTICIPANTS"
            ]
            > 0
        )
    ]
    .copy()
)


eligible_patterns = (
    eligible_pattern_table[
        "MODALITY_PATTERN_BINARY"
    ]
    .tolist()
)


# ------------------------------------------------------------
# Calculate output-specific metrics within each pattern
# ------------------------------------------------------------

pattern_performance_rows = []


for pattern_name in eligible_patterns:

    pattern_mask = (
        pooled_predictions_with_patterns[
            "MODALITY_PATTERN_BINARY"
        ]
        == pattern_name
    )


    pattern_table = (
        pooled_predictions_with_patterns.loc[
            pattern_mask
        ]
        .copy()
    )


    pattern_targets = (
        pattern_table[
            "TARGET"
        ]
        .to_numpy(
            dtype=int
        )
    )


    participant_count = int(
        len(
            pattern_table
        )
    )


    sMCI_count = int(
        np.sum(
            pattern_targets
            == 0
        )
    )


    pMCI_count = int(
        np.sum(
            pattern_targets
            == 1
        )
    )


    pMCI_prevalence = float(
        np.mean(
            pattern_targets
        )
    )


    available_modalities = (
        pattern_table[
            "AVAILABLE_MODALITIES"
        ]
        .iloc[
            0
        ]
    )


    missing_modalities = (
        pattern_table[
            "MISSING_MODALITIES"
        ]
        .iloc[
            0
        ]
    )


    modality_count = int(
        pattern_table[
            "RECONSTRUCTED_MODALITY_COUNT"
        ]
        .iloc[
            0
        ]
    )


    for output_name in OUTPUT_ORDER:

        probability_column = (
            POOLED_OUTPUT_COLUMNS[
                output_name
            ][
                "probability"
            ]
        )


        uncertainty_column = (
            POOLED_OUTPUT_COLUMNS[
                output_name
            ][
                "uncertainty"
            ]
        )


        pattern_probabilities = (
            pattern_table[
                probability_column
            ]
            .to_numpy(
                dtype=float
            )
        )


        pattern_uncertainties = (
            pattern_table[
                uncertainty_column
            ]
            .to_numpy(
                dtype=float
            )
        )


        pattern_predictions = (
            pattern_probabilities
            >= CLASSIFICATION_THRESHOLD
        ).astype(
            int
        )


        pattern_confusion = confusion_matrix(
            pattern_targets,
            pattern_predictions,
            labels=[
                0,
                1,
            ],
        )


        true_negative = int(
            pattern_confusion[
                0,
                0
            ]
        )


        false_positive = int(
            pattern_confusion[
                0,
                1
            ]
        )


        false_negative = int(
            pattern_confusion[
                1,
                0
            ]
        )


        true_positive = int(
            pattern_confusion[
                1,
                1
            ]
        )


        sensitivity = float(
            true_positive
            /
            (
                true_positive
                + false_negative
            )
        )


        specificity = float(
            true_negative
            /
            (
                true_negative
                + false_positive
            )
        )


        roc_auc = float(
            roc_auc_score(
                pattern_targets,
                pattern_probabilities,
            )
        )


        average_precision = float(
            average_precision_score(
                pattern_targets,
                pattern_probabilities,
            )
        )


        balanced_accuracy = float(
            balanced_accuracy_score(
                pattern_targets,
                pattern_predictions,
            )
        )


        accuracy = float(
            accuracy_score(
                pattern_targets,
                pattern_predictions,
            )
        )


        pattern_performance_rows.append(
            {
                "MODALITY_PATTERN_BINARY":
                    pattern_name,

                "MODALITY_COUNT":
                    modality_count,

                "AVAILABLE_MODALITIES":
                    available_modalities,

                "MISSING_MODALITIES":
                    missing_modalities,

                "OUTPUT":
                    output_name,

                "PARTICIPANTS":
                    participant_count,

                "sMCI_PARTICIPANTS":
                    sMCI_count,

                "pMCI_PARTICIPANTS":
                    pMCI_count,

                "pMCI_PREVALENCE":
                    pMCI_prevalence,

                "ROC_AUC":
                    roc_auc,

                "AVERAGE_PRECISION":
                    average_precision,

                "ACCURACY":
                    accuracy,

                "BALANCED_ACCURACY":
                    balanced_accuracy,

                "SENSITIVITY":
                    sensitivity,

                "SPECIFICITY":
                    specificity,

                "BRIER_SCORE":
                    float(
                        brier_score_loss(
                            pattern_targets,
                            pattern_probabilities,
                        )
                    ),

                "ERROR_RATE":
                    float(
                        1.0
                        - accuracy
                    ),

                "MEAN_UNCERTAINTY":
                    float(
                        np.mean(
                            pattern_uncertainties
                        )
                    ),

                "MEDIAN_UNCERTAINTY":
                    float(
                        np.median(
                            pattern_uncertainties
                        )
                    ),
            }
        )


pattern_performance_summary = pd.DataFrame(
    pattern_performance_rows
)


# ------------------------------------------------------------
# Preserve clear output ordering
# ------------------------------------------------------------

pattern_performance_summary[
    "OUTPUT"
] = pd.Categorical(
    pattern_performance_summary[
        "OUTPUT"
    ],
    categories=OUTPUT_ORDER,
    ordered=True,
)


pattern_performance_summary = (
    pattern_performance_summary
    .sort_values(
        [
            "PARTICIPANTS",
            "MODALITY_PATTERN_BINARY",
            "OUTPUT",
        ],
        ascending=[
            False,
            True,
            True,
        ],
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Summarise Hybrid gate weights within each eligible pattern
# ------------------------------------------------------------

hybrid_gate_by_pattern = (
    pooled_predictions_with_patterns.loc[
        pooled_predictions_with_patterns[
            "MODALITY_PATTERN_BINARY"
        ]
        .isin(
            eligible_patterns
        )
    ]
    .groupby(
        [
            "MODALITY_PATTERN_BINARY",
            "RECONSTRUCTED_MODALITY_COUNT",
            "AVAILABLE_MODALITIES",
            "MISSING_MODALITIES",
        ],
        dropna=False,
    )
    .agg(
        PARTICIPANTS=(
            "RID",
            "size",
        ),

        MEAN_3MT_WEIGHT=(
            "W_3MT",
            "mean",
        ),

        SD_3MT_WEIGHT=(
            "W_3MT",
            "std",
        ),

        MEAN_TMC_WEIGHT=(
            "W_TMC",
            "mean",
        ),

        SD_TMC_WEIGHT=(
            "W_TMC",
            "std",
        ),

        MEAN_HYBRID_UNCERTAINTY=(
            "FINAL_UNCERTAINTY",
            "mean",
        ),
    )
    .reset_index()
    .sort_values(
        [
            "PARTICIPANTS",
            "MODALITY_PATTERN_BINARY",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Record excluded patterns and reasons
# ------------------------------------------------------------

excluded_pattern_table = (
    exact_pattern_distribution.loc[
        ~exact_pattern_distribution[
            "MODALITY_PATTERN_BINARY"
        ]
        .isin(
            eligible_patterns
        )
    ]
    .copy()
)


excluded_pattern_table[
    "EXCLUSION_REASON"
] = np.where(
    excluded_pattern_table[
        "PARTICIPANTS"
    ]
    < MINIMUM_PATTERN_PARTICIPANTS,
    (
        "Fewer than "
        + str(
            MINIMUM_PATTERN_PARTICIPANTS
        )
        + " participants"
    ),
    "Only one outcome class represented",
)


# ------------------------------------------------------------
# Define and create output directories
# ------------------------------------------------------------

TABLE_DIR = (
    AGGREGATION_ROOT
    / "tables"
)


TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# Save pattern-level analysis tables
# ------------------------------------------------------------

PATTERN_PERFORMANCE_SUMMARY_PATH = (
    TABLE_DIR
    / "performance_by_exact_modality_pattern.csv"
)


HYBRID_GATE_BY_PATTERN_PATH = (
    TABLE_DIR
    / "hybrid_gate_by_exact_modality_pattern.csv"
)


EXCLUDED_PATTERN_SUMMARY_PATH = (
    TABLE_DIR
    / "excluded_exact_modality_patterns.csv"
)


pattern_performance_summary.to_csv(
    PATTERN_PERFORMANCE_SUMMARY_PATH,
    index=False,
)


hybrid_gate_by_pattern.to_csv(
    HYBRID_GATE_BY_PATTERN_PATH,
    index=False,
)


excluded_pattern_table.to_csv(
    EXCLUDED_PATTERN_SUMMARY_PATH,
    index=False,
)


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("=" * 72)
print("PERFORMANCE BY EXACT MODALITY PATTERN")
print("=" * 72)


print(
    "\nEligible patterns:"
)

print(
    len(
        eligible_patterns
    )
)


print(
    "\nPattern-level performance:"
)

display(
    pattern_performance_summary[
        [
            "MODALITY_PATTERN_BINARY",
            "MISSING_MODALITIES",
            "OUTPUT",
            "PARTICIPANTS",
            "sMCI_PARTICIPANTS",
            "pMCI_PARTICIPANTS",
            "ROC_AUC",
            "AVERAGE_PRECISION",
            "BALANCED_ACCURACY",
            "SENSITIVITY",
            "SPECIFICITY",
            "BRIER_SCORE",
            "MEAN_UNCERTAINTY",
        ]
    ]
    .round(
        6
    )
)


print(
    "\nHybrid gate behaviour by eligible modality pattern:"
)

display(
    hybrid_gate_by_pattern[
        [
            "MODALITY_PATTERN_BINARY",
            "MISSING_MODALITIES",
            "PARTICIPANTS",
            "MEAN_3MT_WEIGHT",
            "SD_3MT_WEIGHT",
            "MEAN_TMC_WEIGHT",
            "SD_TMC_WEIGHT",
            "MEAN_HYBRID_UNCERTAINTY",
        ]
    ]
    .round(
        6
    )
)


print(
    "\nPatterns excluded from standalone performance analysis:"
)

display(
    excluded_pattern_table[
        [
            "MODALITY_PATTERN_BINARY",
            "MISSING_MODALITIES",
            "PARTICIPANTS",
            "sMCI_PARTICIPANTS",
            "pMCI_PARTICIPANTS",
            "EXCLUSION_REASON",
        ]
    ]
)


print(
    "\nPattern-level performance saved to:\n"
    f"{PATTERN_PERFORMANCE_SUMMARY_PATH}"
)


print(
    "\nHybrid gate-by-pattern summary saved to:\n"
    f"{HYBRID_GATE_BY_PATTERN_PATH}"
)


print(
    "\nExcluded-pattern summary saved to:\n"
    f"{EXCLUDED_PATTERN_SUMMARY_PATH}"
)

## 1.15. Final five-fold result

This final cell extracts the pooled Hybrid ROC AUC and the corresponding fold-level mean ± standard deviation for direct reporting.


In [ ]:
# ============================================================
# 15. Reporting the final five-fold Hybrid ROC AUC
# ============================================================

pooled_hybrid_row = (
    pooled_performance_summary.loc[
        pooled_performance_summary[
            "OUTPUT"
        ]
        == "Hybrid"
    ]
    .iloc[
        0
    ]
)

fold_hybrid_auc_row = (
    cross_validation_summary_long.loc[
        (
            cross_validation_summary_long[
                "OUTPUT"
            ]
            == "Hybrid"
        )
        &
        (
            cross_validation_summary_long[
                "METRIC"
            ]
            == "ROC_AUC"
        )
    ]
    .iloc[
        0
    ]
)


final_result_table = pd.DataFrame(
    [
        {
            "EXPERIMENT":
                EXPERIMENT_NAME,

            "PARTICIPANTS":
                int(
                    len(
                        pooled_out_of_fold_predictions
                    )
                ),

            "POOLED_HYBRID_ROC_AUC":
                float(
                    pooled_hybrid_row[
                        "ROC_AUC"
                    ]
                ),

            "FOLD_MEAN_HYBRID_ROC_AUC":
                float(
                    fold_hybrid_auc_row[
                        "MEAN"
                    ]
                ),

            "FOLD_SD_HYBRID_ROC_AUC":
                float(
                    fold_hybrid_auc_row[
                        "STANDARD_DEVIATION"
                    ]
                ),

            "FOLD_MEAN_PLUS_MINUS_SD":
                fold_hybrid_auc_row[
                    "MEAN_PLUS_MINUS_SD"
                ],
        }
    ]
)


FINAL_RESULT_PATH = (
    TABLE_DIR
    / "final_temporal_prebaseline_result.csv"
)

final_result_table.to_csv(
    FINAL_RESULT_PATH,
    index=False,
)


print("=" * 72)
print("FINAL PRE-BASELINE TEMPORAL-SENSITIVITY RESULT")
print("=" * 72)

display(
    final_result_table.round(
        6
    )
)

print(
    "\nFinal pooled Hybrid ROC AUC:"
)

print(
    f"  {final_result_table.loc[0, 'POOLED_HYBRID_ROC_AUC']:.6f}"
)

print(
    "\nFold-level Hybrid ROC AUC:"
)

print(
    "  "
    + final_result_table.loc[
        0,
        "FOLD_MEAN_PLUS_MINUS_SD"
    ]
)

print(
    "\nFinal result saved to:\n"
    f"{FINAL_RESULT_PATH}"
)
